# House Keeping


In [1]:
import sys
import os
sys.path.append('/scratch_net/ken/radjoe/Projects/Experiments/SAMEXP/')

import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats 
from scipy.stats import shapiro , kstest, mannwhitneyu, ttest_rel, wilcoxon
import torch
from torch import nn
from datasets.datasets import MRIDataset, STAREDataset

from helper_func.analyis_helper import clean_df, plot_boxplots, visualize, generate_batch_views, visualize_views, visualize_class_centroids, visualize_feats, prod_feats
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

## Load 

# ENT

In [2]:
tta_ent_lr4_paths =    "/scratch-second/TTA_results/val_ent_only_decoder_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_only_decoder_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_lr5_paths =   "/scratch-second/TTA_results/val_ent_only_decoder_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_only_decoder_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_lr6_paths =   "/scratch-second/TTA_results/val_ent_only_decoder_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_only_decoder_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [3]:
tta_ent_lr4 = pd.read_csv(tta_ent_lr4_paths)
tta_ent_lr4 = clean_df(tta_ent_lr4, True, 'brats')
tta_ent_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000,37.000000
mean,0.347451,0.529055,0.361748,39.812775,26.543589,36.969618,0.346090,0.537855,0.319247,0.487768,0.605595,0.607594
std,0.371348,0.279651,0.295954,30.095002,25.496432,32.696308,0.364334,0.302368,0.289091,0.427576,0.306614,0.350282
min,0.000000,0.000000,0.000000,3.316625,4.123106,3.605551,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.395182,0.063064,8.544003,7.667994,8.306623,0.000000,0.294491,0.032583,0.000000,0.403769,0.391892
50%,0.130632,0.647008,0.349116,35.158924,15.000000,21.840330,0.245542,0.639732,0.288799,0.554728,0.701065,0.714340
75%,0.743536,0.751518,0.660362,59.260021,35.000000,59.134590,0.728464,0.745572,0.565923,0.897251,0.890986,0.862372
max,0.902239,0.877899,0.802354,105.318565,106.230881,109.844437,0.920353,0.994188,0.867210,1.000000,0.975760,1.000000


In [4]:
tta_ent_lr4_augs = pd.read_csv(tta_ent_lr4_augs_paths)
tta_ent_lr4_augs = clean_df(tta_ent_lr4_augs, True, 'brats')
tta_ent_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
mean,0.010812,0.246477,0.353590,41.442846,49.747759,33.351003,0.005499,0.246850,0.362935,0.793240,0.748994,0.723692
std,0.017241,0.353372,0.338262,21.349903,28.470728,30.821951,0.008813,0.383207,0.378123,0.399336,0.292934,0.197725
min,0.000000,0.006590,0.006326,14.163321,11.000000,6.708204,0.000000,0.003347,0.003174,0.000000,0.213115,0.390689
25%,0.001135,0.009662,0.060952,27.270649,28.212090,9.261352,0.000568,0.004860,0.032195,0.823918,0.682199,0.642544
50%,0.002118,0.031017,0.319631,38.910572,58.287169,21.500000,0.001060,0.015827,0.304153,0.995662,0.840990,0.761863
75%,0.012492,0.509334,0.644071,59.359116,66.254204,55.778839,0.006295,0.417769,0.667136,1.000000,0.948070,0.868952
max,0.043880,0.738922,0.749011,67.106636,83.815269,78.010895,0.022432,0.893146,0.836459,1.000000,0.985836,0.920319


In [5]:
tta_ent_lr4[tta_ent_lr4.name.isin(tta_ent_lr4_augs.name)].describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.511862,0.602052,0.578263,31.214291,34.640676,27.504778,0.480896,0.644672,0.523883,0.750095,0.605974,0.680441
std,0.466256,0.267663,0.187756,33.328441,29.772127,34.159315,0.440215,0.256161,0.234938,0.425755,0.325646,0.176743
min,0.000000,0.134655,0.337774,7.000000,7.071068,8.306623,0.000000,0.231380,0.292283,0.000000,0.094958,0.400036
25%,0.004504,0.647008,0.427645,7.280110,14.142136,8.660254,0.002257,0.639732,0.321060,0.811717,0.495545,0.640167
50%,0.821408,0.680859,0.652402,19.467922,23.579653,13.152946,0.728464,0.692940,0.524795,0.967567,0.727638,0.717895
75%,0.831161,0.753378,0.687974,35.158924,48.476799,19.313208,0.831333,0.727492,0.614069,0.971191,0.781174,0.782103
max,0.902239,0.794359,0.785520,87.164497,79.933723,88.090858,0.842429,0.931815,0.867210,1.000000,0.930556,0.862003


In [6]:
tta_ent_lr5 = pd.read_csv(tta_ent_lr5_paths)
tta_ent_lr5 = clean_df(tta_ent_lr5, True, 'brats')
tta_ent_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.345950,0.592454,0.383468,33.461456,25.006452,28.488149,0.360932,0.682188,0.324101,0.507619,0.549770,0.701744
std,0.322100,0.226869,0.283075,28.485880,22.077540,28.299003,0.332153,0.257898,0.277160,0.404473,0.218328,0.262484
min,0.000000,0.017171,0.000000,3.741657,4.123106,4.000000,0.000000,0.011720,0.000000,0.000000,0.032103,0.000000
25%,0.036186,0.448768,0.094511,10.331800,8.802195,9.273154,0.022189,0.545544,0.050700,0.025116,0.423536,0.657176
50%,0.269351,0.637447,0.384779,22.924435,15.890470,12.869825,0.323034,0.760809,0.252649,0.593084,0.575361,0.743052
75%,0.628172,0.782917,0.662256,51.824095,33.051053,41.366741,0.662634,0.886646,0.583633,0.908269,0.730790,0.882866
max,0.890074,0.912230,0.835890,102.182434,77.440300,97.804138,0.972318,0.989233,0.800254,1.000000,0.896855,1.000000


In [7]:
tta_ent_lr5_augs = pd.read_csv(tta_ent_lr5_augs_paths)
tta_ent_lr5_augs = clean_df(tta_ent_lr5_augs, True, 'brats')
tta_ent_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.223219,0.350183,0.322123,31.737899,38.053950,35.638066,0.171668,0.294377,0.259953,0.797787,0.649057,0.704682
std,0.237969,0.262357,0.274939,24.992261,26.658249,28.248117,0.208676,0.229223,0.245100,0.311777,0.316316,0.276847
min,0.000000,0.000000,0.000000,5.472621,8.062258,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.007039,0.115547,0.053527,18.362081,22.000000,17.645781,0.003532,0.062846,0.027874,0.839205,0.551706,0.647165
50%,0.147397,0.359445,0.323897,26.009612,30.398997,26.152507,0.080284,0.263899,0.224220,0.921396,0.741694,0.765594
75%,0.363707,0.549017,0.597364,36.303587,45.209824,38.673186,0.263313,0.476664,0.465049,0.996982,0.883605,0.873105
max,0.654022,0.775618,0.736495,105.706200,99.281654,107.916634,0.562953,0.671046,0.672708,1.000000,0.933687,0.975224


In [8]:
tta_ent_lr5[tta_ent_lr5.name.isin(tta_ent_lr5_augs.name)].describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.436831,0.646186,0.465845,27.743926,25.368438,25.538963,0.378363,0.773265,0.410045,0.626214,0.574322,0.634619
std,0.321974,0.224161,0.253336,22.673573,22.605008,26.192229,0.319517,0.215890,0.262921,0.356594,0.224896,0.251273
min,0.000000,0.017171,0.000000,3.741657,4.242640,4.000000,0.000000,0.011720,0.000000,0.000000,0.032103,0.000000
25%,0.142331,0.562599,0.223653,8.734686,9.378971,9.380372,0.095890,0.744511,0.143690,0.425894,0.443777,0.538473
50%,0.490177,0.704799,0.551769,19.960443,14.676350,12.863961,0.337108,0.858542,0.488939,0.822953,0.641020,0.724271
75%,0.739790,0.803157,0.662738,43.624279,30.696054,25.881215,0.687618,0.892129,0.588248,0.911926,0.737418,0.781248
max,0.890074,0.869245,0.749329,74.408989,75.099930,87.212387,0.885854,0.965970,0.800254,0.988251,0.826542,0.944659


In [9]:
tta_ent_lr6 = pd.read_csv(tta_ent_lr6_paths)
tta_ent_lr6 = clean_df(tta_ent_lr6, True, 'brats')
tta_ent_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.361895,0.571142,0.378368,34.391520,31.673494,27.771651,0.410553,0.768975,0.321725,0.446534,0.472960,0.680900
std,0.309354,0.197243,0.281853,31.673580,23.183794,25.071039,0.320555,0.222892,0.277149,0.391803,0.190151,0.282611
min,0.000000,0.118221,0.000000,4.123106,4.242640,4.000000,0.000000,0.107792,0.000000,0.000000,0.085574,0.000000
25%,0.000396,0.442053,0.072520,9.066633,10.637367,9.041539,0.008427,0.653337,0.038209,0.000203,0.348972,0.560533
50%,0.387726,0.598615,0.377712,23.476081,23.076038,17.185967,0.455292,0.844677,0.258887,0.376592,0.469729,0.738138
75%,0.608960,0.735212,0.629279,54.481959,51.146341,41.056589,0.699035,0.931210,0.545880,0.837624,0.613498,0.904836
max,0.886577,0.826865,0.842670,110.330559,78.185677,92.598595,0.972318,0.989781,0.838721,1.000000,0.820519,1.000000


In [10]:
tta_ent_lr6_augs = pd.read_csv(tta_ent_lr6_augs_paths)
tta_ent_lr6_augs = clean_df(tta_ent_lr6_augs, True, 'brats')
tta_ent_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.238055,0.388731,0.343354,26.633211,32.571324,33.203225,0.205588,0.364586,0.281225,0.715358,0.677753,0.684797
std,0.237385,0.285223,0.296811,19.560216,21.616232,28.258712,0.256517,0.286907,0.266748,0.300183,0.256163,0.303059
min,0.000000,0.000000,0.000000,5.656854,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.016291,0.129670,0.032627,14.085904,18.552486,16.699083,0.008292,0.071395,0.016601,0.582601,0.575881,0.645098
50%,0.261503,0.351322,0.291641,23.493559,28.000000,25.632011,0.162148,0.428741,0.195855,0.856793,0.797518,0.773943
75%,0.431053,0.638474,0.622667,28.041750,36.194920,31.388412,0.292480,0.565019,0.521602,0.918430,0.861024,0.880362
max,0.683188,0.865446,0.782659,78.651131,90.555222,107.029198,0.868949,0.845559,0.779944,1.000000,0.889044,0.993228


In [11]:
tta_ent_lr6[tta_ent_lr6.name.isin(tta_ent_lr6_augs.name)].describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.446871,0.627680,0.449259,27.804746,26.277991,27.167576,0.416537,0.831520,0.392521,0.564753,0.519712,0.656441
std,0.312492,0.192209,0.271224,29.640232,21.656164,25.515496,0.316517,0.187870,0.276062,0.368361,0.181485,0.265304
min,0.000000,0.118221,0.000000,4.123106,4.242640,4.000000,0.000000,0.107792,0.000000,0.000000,0.085574,0.000000
25%,0.177941,0.547042,0.200971,7.245444,10.278792,9.163247,0.111897,0.799181,0.117814,0.248773,0.426816,0.564547
50%,0.526267,0.680911,0.571231,12.310039,15.377861,16.717444,0.430351,0.882688,0.459227,0.715087,0.557183,0.713621
75%,0.659101,0.747868,0.655078,43.102950,42.820893,31.340696,0.689805,0.934385,0.602508,0.847085,0.669478,0.831935
max,0.886577,0.826865,0.823427,104.795036,70.007141,87.069519,0.899493,0.981111,0.825060,0.999330,0.741959,0.945113


In [12]:
tta_ent_norm_lr4_paths =    "/scratch-second/TTA_results/val_ent_norm_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_norm_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_norm_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_norm_lr5_paths =   "/scratch-second/TTA_results/val_ent_norm_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_norm_lr5_augs_paths =   "/scratch-second/TTA_results/val_ent_norm_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_norm_lr6_paths =   "/scratch-second/TTA_results/val_ent_norm_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_norm_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_norm_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [13]:
tta_ent_norm_lr4 = pd.read_csv(tta_ent_norm_lr4_paths)
tta_ent_norm_lr4 = clean_df(tta_ent_norm_lr4, True, 'brats')
tta_ent_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000,36.000000
mean,0.368338,0.521420,0.349639,36.408251,29.644361,39.939850,0.338532,0.529913,0.288294,0.527913,0.610096,0.601889
std,0.355535,0.272544,0.279313,30.625315,24.368846,37.611978,0.352965,0.305107,0.258044,0.431502,0.303047,0.347061
min,0.000000,0.000000,0.000000,3.316625,4.123106,3.605551,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.345271,0.044682,9.962214,11.022633,8.900581,0.000000,0.243574,0.025085,0.000000,0.427977,0.429336
50%,0.380140,0.608205,0.381917,31.092156,16.613679,21.509996,0.252555,0.624433,0.245940,0.611561,0.695643,0.742156
75%,0.724309,0.730204,0.613210,55.106926,49.512837,67.943756,0.674487,0.760177,0.491468,0.942756,0.843597,0.872319
max,0.869658,0.861298,0.805544,121.282104,79.914955,111.933014,0.964775,0.994116,0.831592,1.000000,0.998294,1.000000


In [14]:
tta_ent_norm_lr4_augs = pd.read_csv(tta_ent_norm_lr4_augs_paths)
tta_ent_norm_lr4_augs = clean_df(tta_ent_norm_lr4_augs, True, 'brats')
tta_ent_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
mean,0.000791,0.182559,0.348675,59.014535,53.523505,47.108149,0.000413,0.125154,0.339954,0.606385,0.581725,0.384908
std,0.000993,0.193316,0.302655,10.028826,19.564674,32.692605,0.000526,0.134678,0.302677,0.518294,0.448959,0.359327
min,0.000131,0.063350,0.000000,48.600410,36.715118,11.000000,0.000066,0.041088,0.000000,0.019156,0.076854,0.000000
25%,0.000220,0.071035,0.251264,54.218012,42.785257,33.312527,0.000110,0.047485,0.219834,0.409578,0.404533,0.221600
50%,0.000309,0.078721,0.502527,59.835613,48.855396,55.625053,0.000155,0.053882,0.439669,0.800000,0.732212,0.443199
75%,0.001121,0.242163,0.523012,64.221598,61.927698,65.162224,0.000586,0.167186,0.509931,0.900000,0.834161,0.577363
max,0.001933,0.405605,0.543498,68.607582,75.000000,74.699394,0.001018,0.280490,0.580193,1.000000,0.936110,0.711526


In [15]:
tta_ent_norm_lr5 = pd.read_csv(tta_ent_norm_lr5_paths)
tta_ent_norm_lr5 = clean_df(tta_ent_norm_lr5, True, 'brats')
tta_ent_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.356890,0.595691,0.390536,30.263813,26.559309,26.540160,0.372795,0.694666,0.328849,0.526094,0.544592,0.699405
std,0.322078,0.223720,0.280808,27.014532,23.359292,25.885800,0.333781,0.257708,0.275470,0.408580,0.213759,0.258183
min,0.000000,0.014883,0.000000,3.741657,4.000000,3.741657,0.000000,0.009771,0.000000,0.000000,0.031211,0.000000
25%,0.047059,0.471513,0.117709,9.055386,8.774964,9.000000,0.026440,0.575540,0.067199,0.028143,0.428184,0.666675
50%,0.303201,0.667576,0.390052,20.619162,16.911535,13.000000,0.352334,0.765470,0.262323,0.585053,0.597457,0.755212
75%,0.663833,0.781776,0.655017,42.767982,44.944408,39.344631,0.679373,0.892779,0.580807,0.915512,0.705071,0.827129
max,0.891719,0.911349,0.835756,98.208961,78.262383,96.273048,0.973472,0.990575,0.792899,1.000000,0.891901,1.000000


In [16]:
tta_ent_norm_lr5_augs = pd.read_csv(tta_ent_norm_lr5_augs_paths)
tta_ent_norm_lr5_augs = clean_df(tta_ent_norm_lr5_augs, True, 'brats')
tta_ent_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.231493,0.367464,0.341616,32.429403,38.491259,35.432068,0.179716,0.308362,0.280662,0.784430,0.635039,0.700413
std,0.239896,0.252798,0.277046,25.643739,26.918278,29.382018,0.213151,0.221360,0.253786,0.317327,0.316674,0.284598
min,0.000000,0.000000,0.000000,5.826631,9.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.014062,0.160929,0.068206,16.599504,19.348469,16.851359,0.007082,0.099884,0.035812,0.832284,0.527674,0.615987
50%,0.162754,0.377425,0.382098,26.267851,31.016125,24.269321,0.088586,0.281127,0.312726,0.919897,0.717659,0.767808
75%,0.395805,0.547857,0.601106,37.999039,45.967384,44.441107,0.326371,0.490764,0.477742,0.972777,0.880503,0.876769
max,0.658953,0.775782,0.736208,105.777122,98.234413,107.916634,0.560535,0.671000,0.685868,1.000000,0.921121,0.974110


In [17]:
tta_ent_norm_lr6 = pd.read_csv(tta_ent_norm_lr6_paths)
tta_ent_norm_lr6 = clean_df(tta_ent_norm_lr6, True, 'brats')
tta_ent_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000,43.000000
mean,0.361006,0.581592,0.385895,33.422249,31.938739,27.694809,0.412626,0.784677,0.328257,0.444360,0.480643,0.675875
std,0.313264,0.186444,0.280865,31.408536,23.402368,25.364717,0.324356,0.200538,0.276441,0.396510,0.184677,0.283120
min,0.000000,0.147195,0.000000,4.123106,4.242640,4.000000,0.000000,0.146466,0.000000,0.000000,0.085520,0.000000
25%,0.000263,0.476703,0.087699,8.752435,10.458079,9.027693,0.005618,0.679605,0.049194,0.000135,0.357147,0.560837
50%,0.373759,0.600768,0.386950,22.825424,23.759205,16.881943,0.467263,0.847311,0.275853,0.348327,0.499394,0.734150
75%,0.611452,0.736950,0.635053,51.330475,51.805061,43.325684,0.700665,0.933310,0.546654,0.845283,0.615705,0.885828
max,0.886721,0.826597,0.842641,110.330559,78.115303,92.598595,0.972318,0.989876,0.838903,1.000000,0.820392,1.000000


In [18]:
tta_ent_norm_lr6_augs = pd.read_csv(tta_ent_norm_lr6_augs_paths)
tta_ent_norm_lr6_augs = clean_df(tta_ent_norm_lr6_augs, True, 'brats')
tta_ent_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.237707,0.391965,0.349685,26.670018,32.361975,32.799416,0.205571,0.367767,0.290337,0.715165,0.678458,0.679842
std,0.237816,0.284518,0.304633,19.571865,21.701232,28.480823,0.257358,0.285642,0.278393,0.299837,0.254730,0.300608
min,0.000000,0.000000,0.000000,5.486172,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.016614,0.129863,0.034166,14.139334,18.552486,15.427499,0.008457,0.071461,0.017400,0.582440,0.573508,0.643648
50%,0.258060,0.355912,0.292239,23.600847,28.000000,25.159491,0.162981,0.429012,0.196411,0.857835,0.796850,0.767224
75%,0.429887,0.638863,0.622941,27.596541,36.194920,30.998334,0.291775,0.564928,0.523155,0.917453,0.859625,0.861947
max,0.684353,0.865285,0.798710,78.683228,90.555222,107.029198,0.871901,0.845341,0.779518,1.000000,0.888734,0.993304


In [19]:
tta_ent_first_layer_lr4_paths =    "/scratch-second/TTA_results/val_ent_first_layer_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_first_layer_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_first_layer_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_first_layer_lr5_paths =   "/scratch-second/TTA_results/val_ent_first_layer_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_first_layer_lr5_augs_paths =   "/scratch-second/TTA_results/val_ent_first_layer_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_first_layer_lr6_paths =   "/scratch-second/TTA_results/val_ent_first_layer_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_ent_first_layer_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_first_layer_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [20]:
tta_ent_first_layer_lr4 = pd.read_csv(tta_ent_first_layer_lr4_paths)
tta_ent_first_layer_lr4 = clean_df(tta_ent_first_layer_lr4, True, 'brats')
tta_ent_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000,32.000000
mean,0.370264,0.532518,0.357356,29.279206,28.805231,34.617266,0.343674,0.554454,0.289115,0.621467,0.610536,0.639368
std,0.351058,0.261715,0.259623,22.734954,22.831730,33.530667,0.360103,0.305476,0.243649,0.412472,0.281688,0.317419
min,0.000000,0.000000,0.000000,4.000000,6.403124,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.002606,0.416793,0.116816,9.632658,11.339414,9.420011,0.003238,0.296198,0.062195,0.217723,0.449631,0.549215
50%,0.234043,0.607510,0.346229,23.692251,20.082926,16.199719,0.143098,0.666911,0.215146,0.866192,0.678517,0.727798
75%,0.783239,0.752998,0.594022,47.104850,41.824518,57.591693,0.731826,0.775061,0.505135,0.983098,0.835440,0.856151
max,0.861580,0.851679,0.793755,80.432762,79.937477,116.321968,0.968293,0.995545,0.855805,1.000000,0.976249,1.000000


In [21]:
tta_ent_first_layer_lr4_augs = pd.read_csv(tta_ent_first_layer_lr4_augs_paths)
tta_ent_first_layer_lr4_augs = clean_df(tta_ent_first_layer_lr4_augs, True, 'brats')
tta_ent_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000
mean,0.201722,0.206878,0.410459,34.774672,52.365384,42.151870,0.153385,0.135712,0.357588,0.969982,0.621637,0.637299
std,0.314309,0.204362,0.204965,18.433400,21.067348,29.850872,0.267450,0.137863,0.226831,0.031785,0.413662,0.148950
min,0.000516,0.031325,0.157840,7.348469,29.698484,10.677078,0.000258,0.016440,0.091018,0.930551,0.054780,0.405194
25%,0.026050,0.052948,0.248220,26.000000,37.023640,27.000000,0.013198,0.051234,0.150014,0.941720,0.331311,0.593743
50%,0.045321,0.129934,0.472767,40.111088,51.282059,29.615026,0.023219,0.069543,0.410048,0.982612,0.757066,0.673249
75%,0.188040,0.300482,0.509674,46.658333,60.489670,56.956551,0.103968,0.187438,0.567387,0.995025,0.977772,0.718741
max,0.748685,0.519703,0.663795,53.755470,83.333069,86.510696,0.626284,0.353905,0.569471,1.000000,0.987256,0.795568


In [22]:
tta_ent_first_layer_lr5 = pd.read_csv(tta_ent_first_layer_lr5_paths)
tta_ent_first_layer_lr5 = clean_df(tta_ent_first_layer_lr5, True, 'brats')
tta_ent_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.358030,0.584597,0.378982,30.735732,26.534636,27.667209,0.367957,0.680120,0.317048,0.526239,0.536369,0.715117
std,0.325382,0.236523,0.282949,27.960751,23.170720,26.303514,0.334476,0.274911,0.274916,0.414956,0.222665,0.262960
min,0.000000,0.000000,0.000000,3.605551,4.000000,3.741657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.037314,0.453403,0.099344,9.150034,8.831223,9.433981,0.022477,0.545308,0.053914,0.028411,0.402935,0.654596
50%,0.346789,0.640913,0.379340,20.724972,17.058621,14.651063,0.324879,0.772974,0.254813,0.672821,0.571765,0.762135
75%,0.672264,0.777979,0.653130,43.818460,42.557903,37.994378,0.684272,0.895456,0.574140,0.920246,0.700678,0.891392
max,0.892560,0.910547,0.834354,109.682266,78.869514,96.806503,0.974625,0.992393,0.779422,1.000000,0.888232,1.000000


In [23]:
tta_ent_first_layer_lr5_augs = pd.read_csv(tta_ent_first_layer_lr5_augs_paths)
tta_ent_first_layer_lr5_augs = clean_df(tta_ent_first_layer_lr5_augs, True, 'brats')
tta_ent_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.225200,0.369180,0.339911,31.652211,37.295245,34.784225,0.173669,0.310718,0.285118,0.799065,0.647886,0.707487
std,0.242607,0.263444,0.282172,23.585241,26.959609,28.790719,0.210778,0.226863,0.271879,0.312690,0.313889,0.279075
min,0.000000,0.000000,0.000000,5.830952,9.433981,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.008402,0.119536,0.062056,18.656979,17.941206,16.762975,0.004219,0.064294,0.032600,0.859565,0.548359,0.611971
50%,0.121159,0.389967,0.321537,26.387488,29.852700,25.509256,0.064534,0.335939,0.222951,0.922128,0.734233,0.793322
75%,0.412039,0.575662,0.598342,37.433726,45.266598,38.685006,0.300612,0.517221,0.463166,0.996210,0.881513,0.856219
max,0.676016,0.764884,0.779335,96.569145,99.399437,107.790306,0.549662,0.652053,0.776012,1.000000,0.925409,0.977558


In [24]:
tta_ent_first_layer_lr6 = pd.read_csv(tta_ent_first_layer_lr6_paths)
tta_ent_first_layer_lr6 = clean_df(tta_ent_first_layer_lr6, True, 'brats')
tta_ent_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.362689,0.571172,0.377961,34.353417,31.688221,27.755642,0.410686,0.769081,0.320775,0.446762,0.472902,0.681814
std,0.310205,0.197106,0.282070,31.704112,23.214765,25.082291,0.321145,0.223156,0.276686,0.392123,0.189938,0.282092
min,0.000000,0.117886,0.000000,4.123106,4.242640,4.000000,0.000000,0.106754,0.000000,0.000000,0.085490,0.000000
25%,0.000395,0.442205,0.073760,8.945668,10.532685,9.041539,0.008427,0.655880,0.038873,0.000202,0.349227,0.563188
50%,0.388151,0.600833,0.377980,23.506099,23.102479,17.185967,0.461599,0.844817,0.261168,0.379014,0.469330,0.737694
75%,0.612399,0.736608,0.629684,54.481567,51.146341,41.069678,0.697578,0.931646,0.537722,0.836436,0.612429,0.905448
max,0.887116,0.826871,0.842597,110.330559,78.147293,92.598595,0.972318,0.990066,0.837632,1.000000,0.820562,1.000000


In [25]:
tta_ent_first_layer_lr6_augs = pd.read_csv(tta_ent_first_layer_lr6_augs_paths)
tta_ent_first_layer_lr6_augs = clean_df(tta_ent_first_layer_lr6_augs, True, 'brats')
tta_ent_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.240195,0.393042,0.344694,26.675764,32.489529,33.076061,0.207539,0.367578,0.282051,0.716534,0.677625,0.680208
std,0.239323,0.284002,0.295890,19.572339,21.655150,28.281584,0.257942,0.284770,0.266049,0.299830,0.256250,0.300790
min,0.000000,0.000000,0.000000,5.477226,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.016491,0.129532,0.043485,14.165020,18.604687,16.699083,0.008397,0.071402,0.022297,0.584803,0.575254,0.643224
50%,0.255542,0.351841,0.293310,23.622025,28.000000,25.099800,0.167522,0.428725,0.197555,0.854545,0.796807,0.767864
75%,0.428055,0.638315,0.624214,28.041750,36.194920,30.930914,0.291868,0.565463,0.522774,0.916273,0.867665,0.860497
max,0.689297,0.865291,0.782568,78.555710,90.658699,107.029198,0.873672,0.845097,0.777191,1.000000,0.888323,0.992669


### Commons L-4


In [26]:
common_augs = tta_ent_lr5_augs[
    tta_ent_lr5_augs['name'].isin(tta_ent_first_layer_lr5_augs['name']) &
    tta_ent_lr5_augs['name'].isin(tta_ent_norm_lr5_augs['name'])
]
common_augs

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00007-000.npy,0.654022,0.662733,0.603597,13.000000,24.000000,18.000000,0.517843,0.534871,0.500274,0.887378,0.870931,0.760708
1,BraTS-SSA-00010-000.npy,0.018639,0.213087,0.060911,45.705578,16.000000,28.653097,0.009407,0.221472,0.031849,1.000000,0.205313,0.696123
2,BraTS-SSA-00011-000.npy,0.231972,0.224066,0.073756,24.166092,28.000000,29.427877,0.132329,0.128021,0.038756,0.939148,0.897069,0.761080
3,BraTS-SSA-00015-000.npy,0.605964,0.652698,0.003839,26.000000,31.032242,58.830265,0.467334,0.521811,0.001923,0.861525,0.871232,0.965517
4,BraTS-SSA-00044-000.npy,0.006477,0.085208,0.119948,34.263683,79.392700,88.628448,0.003249,0.044641,0.064571,1.000000,0.933687,0.842362
5,BraTS-SSA-00051-000.npy,0.226586,0.738693,0.736495,5.472621,13.000000,16.583124,0.127768,0.621283,0.656726,1.000000,0.910820,0.838322
6,BraTS-SSA-00055-000.npy,0.488607,0.394787,0.261629,10.619926,28.773220,23.537205,0.500000,0.253233,0.151429,0.477721,0.895179,0.960947
7,BraTS-SSA-00056-000.npy,0.134295,0.504939,0.431844,20.976177,12.083046,13.266500,0.072881,0.538327,0.333108,0.853519,0.475451,0.613772
8,BraTS-SSA-00057-000.npy,0.312110,0.324104,0.595286,26.019224,44.485954,24.269321,0.194094,0.206593,0.453307,0.796262,0.751643,0.866764
9,BraTS-SSA-00095-000.npy,0.000000,0.000000,0.000000,82.425728,99.084305,80.995369,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [27]:
mask = tta_ent_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.234826,0.367067,0.337545,32.124492,38.211680,36.038191,0.180633,0.309084,0.272854,0.787144,0.636915,0.700371
std,0.238602,0.258141,0.273440,25.615594,27.379157,28.963893,0.210400,0.225601,0.244739,0.316566,0.320160,0.283743
min,0.000000,0.000000,0.000000,5.472621,8.062258,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.012932,0.161851,0.067334,16.574720,20.000000,17.291562,0.006517,0.098468,0.035302,0.824890,0.526288,0.636034
50%,0.160500,0.394787,0.386165,26.019224,29.765753,24.269321,0.087688,0.274565,0.297012,0.917782,0.731745,0.761080
75%,0.405340,0.553056,0.599442,37.836498,45.933693,44.543932,0.331320,0.491713,0.476791,0.985167,0.883206,0.879447
max,0.654022,0.775618,0.736495,105.706200,99.281654,107.916634,0.562953,0.671046,0.672708,1.000000,0.933687,0.975224


In [28]:
mask = tta_ent_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.236902,0.387067,0.356223,32.035158,37.413042,35.139411,0.182734,0.326287,0.299320,0.788490,0.635652,0.702849
std,0.243388,0.257884,0.280049,24.167565,27.693076,29.534598,0.212510,0.221831,0.271600,0.317563,0.317554,0.285929
min,0.000000,0.000000,0.000000,5.830952,9.433981,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.019580,0.155337,0.095525,16.771319,16.882411,16.733094,0.009918,0.132560,0.050973,0.850634,0.534595,0.611166
50%,0.133545,0.398675,0.387441,26.774977,28.705400,24.000000,0.071550,0.405720,0.294652,0.919727,0.715219,0.791031
75%,0.431864,0.588804,0.603261,38.747522,46.069729,44.554628,0.354305,0.518950,0.475616,0.974147,0.880500,0.860268
max,0.676016,0.764884,0.779335,96.569145,99.399437,107.790306,0.549662,0.652053,0.776012,1.000000,0.925409,0.977558


In [29]:
mask = tta_ent_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.231493,0.367464,0.341616,32.429403,38.491259,35.432068,0.179716,0.308362,0.280662,0.784430,0.635039,0.700413
std,0.239896,0.252798,0.277046,25.643739,26.918278,29.382018,0.213151,0.221360,0.253786,0.317327,0.316674,0.284598
min,0.000000,0.000000,0.000000,5.826631,9.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.014062,0.160929,0.068206,16.599504,19.348469,16.851359,0.007082,0.099884,0.035812,0.832284,0.527674,0.615987
50%,0.162754,0.377425,0.382098,26.267851,31.016125,24.269321,0.088586,0.281127,0.312726,0.919897,0.717659,0.767808
75%,0.395805,0.547857,0.601106,37.999039,45.967384,44.441107,0.326371,0.490764,0.477742,0.972777,0.880503,0.876769
max,0.658953,0.775782,0.736208,105.777122,98.234413,107.916634,0.560535,0.671000,0.685868,1.000000,0.921121,0.974110


In [30]:
mask = tta_ent_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.251603,0.418085,0.327152,27.667141,33.150188,36.344793,0.220064,0.403811,0.263217,0.711991,0.661555,0.680053
std,0.220293,0.292070,0.290611,21.164428,23.805327,29.974725,0.258886,0.292548,0.253081,0.265027,0.270317,0.325442
min,0.000000,0.000000,0.000000,5.656854,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.026458,0.139796,0.032627,14.085904,16.713624,16.699083,0.013502,0.076051,0.016601,0.582601,0.575881,0.645098
50%,0.270634,0.556494,0.291641,23.280893,27.000000,28.930952,0.173264,0.500754,0.195855,0.817752,0.761203,0.773943
75%,0.431053,0.659162,0.610406,28.797058,41.765463,44.176872,0.292480,0.604571,0.494987,0.882692,0.852248,0.895149
max,0.611418,0.865446,0.733455,78.651131,90.555222,107.029198,0.868949,0.845559,0.655522,1.000000,0.889044,0.993228


In [31]:
mask = tta_ent_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.254081,0.423321,0.328764,27.622403,33.050523,36.190857,0.222063,0.407447,0.264373,0.713594,0.661374,0.674296
std,0.221768,0.290037,0.289560,21.179921,23.851213,30.018017,0.259760,0.289449,0.252600,0.264706,0.270478,0.322578
min,0.000000,0.000000,0.000000,5.477226,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.026424,0.139822,0.043485,14.165020,16.713624,16.699083,0.013484,0.076071,0.022297,0.584803,0.575254,0.643224
50%,0.284442,0.557491,0.293310,23.194826,25.000000,28.728897,0.169625,0.499834,0.197555,0.836550,0.745333,0.767864
75%,0.428055,0.658030,0.611211,28.747951,41.781712,44.176872,0.291868,0.604756,0.495912,0.882789,0.860707,0.876651
max,0.613984,0.865291,0.733734,78.555710,90.658699,107.029198,0.873672,0.845097,0.656655,1.000000,0.888323,0.992669


In [32]:
mask = tta_ent_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_ent_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.251321,0.418292,0.327707,27.651955,33.144553,36.224393,0.220045,0.404077,0.263799,0.711823,0.660626,0.673700
std,0.220529,0.292147,0.290601,21.190812,23.806256,30.005530,0.259686,0.292675,0.253201,0.264600,0.269925,0.322298
min,0.000000,0.000000,0.000000,5.486172,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.026526,0.139864,0.034166,14.139334,16.713624,16.699083,0.013536,0.076097,0.017400,0.582440,0.573508,0.643648
50%,0.271930,0.557344,0.292239,23.216373,27.000000,28.930952,0.169869,0.500533,0.196411,0.820231,0.752114,0.767224
75%,0.429887,0.658993,0.610668,28.795413,41.765463,44.176872,0.291775,0.605016,0.495705,0.882145,0.851495,0.874732
max,0.613278,0.865285,0.733579,78.683228,90.555222,107.029198,0.871901,0.845341,0.656230,1.000000,0.888734,0.993304


In [33]:
common_rows = tta_ent_lr4[
    tta_ent_lr4['name'].isin(tta_ent_first_layer_lr4['name']) &
    tta_ent_lr4['name'].isin(tta_ent_norm_lr4['name'])
]
common_rows

,name,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
0,BraTS-SSA-00007-000.npy,0.836214,0.814517,0.798667,3.316625,17.492855,4.000000,0.920353,0.745572,0.739915,0.766170,0.897513,0.867552
1,BraTS-SSA-00008-000.npy,0.601524,0.562385,0.008116,13.490738,7.071068,29.765753,0.597993,0.640427,0.004074,0.605098,0.501298,1.000000
2,BraTS-SSA-00010-000.npy,0.077655,0.409856,0.000000,51.276688,13.341664,59.287434,0.045289,0.535778,0.000000,0.272145,0.331860,0.000000
3,BraTS-SSA-00011-000.npy,0.828867,0.707196,0.126635,5.385165,61.261734,59.134590,0.783247,0.799928,0.070000,0.880130,0.633730,0.663214
4,BraTS-SSA-00014-000.npy,0.343625,0.833039,0.283292,105.318565,5.830952,100.900444,0.303075,0.777083,0.239773,0.396702,0.897680,0.346112
5,BraTS-SSA-00015-000.npy,0.902239,0.680859,0.427645,7.000000,23.579653,13.152946,0.842429,0.639732,0.321060,0.971191,0.727638,0.640167
6,BraTS-SSA-00025-000.npy,0.683391,0.486718,0.125370,29.012053,8.306623,54.189934,0.889762,0.671546,0.066905,0.554728,0.381672,0.993855
9,BraTS-SSA-00037-000.npy,0.000000,0.796753,0.486454,26.019224,6.708204,5.000000,0.000000,0.717452,0.349672,0.000000,0.895764,0.799000
10,BraTS-SSA-00041-000.npy,0.002284,0.675100,0.110744,19.261360,9.899495,16.093477,0.001144,0.592974,0.058640,1.000000,0.783631,0.993407
11,BraTS-SSA-00044-000.npy,0.831161,0.794359,0.687974,7.280110,79.933723,88.090858,0.728464,0.692940,0.614069,0.967567,0.930556,0.782103


In [34]:
mask = tta_ent_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_ent_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.416293,0.574717,0.381481,33.451890,25.130377,33.779044,0.389969,0.595775,0.335201,0.591363,0.641613,0.669490
std,0.370857,0.236465,0.289416,28.122408,23.115131,30.465938,0.361525,0.266736,0.293624,0.399658,0.270973,0.291082
min,0.000000,0.001457,0.000000,3.316625,5.830952,3.605551,0.000000,0.001092,0.000000,0.000000,0.002188,0.000000
25%,0.002284,0.481846,0.117732,7.000000,8.062258,8.306623,0.001144,0.519349,0.062548,0.272145,0.495545,0.592370
50%,0.469847,0.675100,0.388293,29.012053,14.142136,19.313208,0.317817,0.653722,0.288799,0.766170,0.717174,0.717895
75%,0.821408,0.751518,0.660362,52.858299,27.367865,54.189934,0.768431,0.777083,0.614069,0.947122,0.890986,0.862003
max,0.902239,0.833039,0.802354,105.318565,79.933723,100.900444,0.920353,0.994188,0.867210,1.000000,0.975760,1.000000


In [35]:
mask = tta_ent_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.407559,0.550259,0.352417,28.115332,28.029758,34.964499,0.378589,0.575871,0.284995,0.651272,0.626293,0.650157
std,0.347937,0.241875,0.256056,22.787989,22.794811,33.712529,0.360688,0.285843,0.244287,0.393371,0.269753,0.307440
min,0.000000,0.000920,0.000000,4.000000,6.403124,4.123106,0.000000,0.000785,0.000000,0.000000,0.001109,0.000000
25%,0.049976,0.464196,0.126198,9.444551,11.575837,9.486833,0.025633,0.440590,0.067462,0.330508,0.462155,0.566483
50%,0.344453,0.607858,0.338877,20.832666,18.788294,17.233688,0.210169,0.673080,0.214004,0.869147,0.684753,0.728458
75%,0.799729,0.747989,0.589787,44.463467,33.436508,56.444664,0.751687,0.771527,0.502533,0.982517,0.839734,0.854549
max,0.861580,0.809115,0.793755,80.432762,79.937477,116.321968,0.968293,0.995545,0.855805,1.000000,0.976249,1.000000


In [36]:
mask = tta_ent_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_ent_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.430850,0.558361,0.371225,30.119108,29.345952,36.958831,0.397331,0.578840,0.310308,0.624216,0.642616,0.649974
std,0.345330,0.231471,0.273074,26.840487,24.788702,36.451131,0.351845,0.270839,0.261976,0.398000,0.265197,0.311962
min,0.000000,0.001358,0.000000,3.316625,6.082763,3.605551,0.000000,0.001073,0.000000,0.000000,0.001852,0.000000
25%,0.018018,0.499875,0.156250,9.848858,11.357817,8.124039,0.009148,0.464561,0.084977,0.341537,0.510334,0.522523
50%,0.484128,0.648635,0.399869,20.420578,15.394804,18.000000,0.353613,0.648024,0.266958,0.833442,0.703950,0.755719
75%,0.750035,0.728087,0.615124,42.070778,46.957428,55.982140,0.738839,0.745939,0.562453,0.961760,0.847093,0.870459
max,0.869658,0.785715,0.805544,113.294304,79.914955,111.933014,0.964775,0.994116,0.831592,1.000000,0.998294,1.000000


### L5

In [37]:
mask = tta_ent_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_ent_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.415256,0.632249,0.405995,32.295596,27.180051,28.102250,0.409890,0.742155,0.350815,0.520370,0.581334,0.688902
std,0.347877,0.206364,0.267688,29.668998,23.722911,28.003693,0.338527,0.229589,0.275231,0.411265,0.205079,0.240126
min,0.000000,0.050345,0.000000,3.741657,4.242640,4.000000,0.000000,0.035587,0.000000,0.000000,0.086015,0.000000
25%,0.035287,0.570049,0.203113,9.219544,7.280110,8.306623,0.048910,0.705350,0.115765,0.024157,0.449853,0.647669
50%,0.460610,0.694707,0.391634,21.000000,16.780939,12.727922,0.482261,0.800961,0.253870,0.603177,0.609789,0.734979
75%,0.759245,0.785390,0.662199,52.216850,46.711884,39.824615,0.766724,0.887265,0.587676,0.910609,0.736132,0.804756
max,0.890074,0.893329,0.835890,102.182434,75.099930,89.587669,0.897894,0.989233,0.800254,0.988251,0.861139,1.000000


In [38]:
mask = tta_ent_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.435726,0.629106,0.398415,30.286656,29.591266,28.669972,0.427449,0.749795,0.338835,0.535531,0.568047,0.695164
std,0.347850,0.204073,0.267877,29.775364,24.915609,27.385050,0.339789,0.231094,0.272829,0.421410,0.200486,0.233548
min,0.000000,0.056181,0.000000,3.605551,4.123106,3.741657,0.000000,0.040988,0.000000,0.000000,0.089267,0.000000
25%,0.036184,0.574817,0.179132,7.000000,8.774964,9.433981,0.043366,0.731525,0.099809,0.028986,0.449808,0.638095
50%,0.451830,0.697968,0.381136,18.838774,19.000000,13.490738,0.506766,0.807520,0.257412,0.798237,0.632325,0.750472
75%,0.792077,0.782234,0.649426,44.294468,50.209560,39.370041,0.776265,0.896729,0.574179,0.925078,0.707092,0.814851
max,0.892560,0.835002,0.834354,109.682266,77.960571,87.191742,0.902547,0.992393,0.779422,1.000000,0.856790,1.000000


In [39]:
mask = tta_ent_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_ent_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.421982,0.627021,0.403196,29.828878,30.582658,27.317372,0.419495,0.747781,0.345944,0.531521,0.565356,0.688567
std,0.347700,0.203680,0.267136,28.065922,25.244822,26.849729,0.341022,0.230385,0.274298,0.421700,0.201861,0.232814
min,0.000000,0.048642,0.000000,3.741657,4.123106,3.741657,0.000000,0.038422,0.000000,0.000000,0.066270,0.000000
25%,0.036724,0.573783,0.205116,7.000000,8.944272,7.874008,0.057917,0.723095,0.117074,0.026885,0.446046,0.666675
50%,0.442663,0.694280,0.390052,18.973665,19.390720,13.000000,0.489887,0.804042,0.262323,0.793950,0.614495,0.742897
75%,0.774784,0.781776,0.652277,44.933285,51.662365,39.344631,0.773585,0.892779,0.580807,0.917088,0.705071,0.798983
max,0.891719,0.836199,0.835756,98.208961,78.262383,87.183716,0.897990,0.990575,0.792899,1.000000,0.855982,1.000000


In [40]:
mask = tta_ent_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.425167,0.612132,0.400137,33.933304,32.230642,28.466473,0.452828,0.811172,0.353140,0.491214,0.512712,0.647597
std,0.335581,0.179398,0.265427,34.264506,25.084560,26.204569,0.325639,0.192529,0.278050,0.409405,0.184905,0.250159
min,0.000000,0.146999,0.000000,4.123106,4.242640,4.000000,0.000000,0.145911,0.000000,0.000000,0.085490,0.000000
25%,0.037915,0.553068,0.157990,7.702074,8.870302,9.243416,0.102062,0.777387,0.089360,0.020137,0.398363,0.562703
50%,0.451155,0.675000,0.415272,19.104973,22.449944,16.881943,0.490689,0.880852,0.329390,0.613106,0.541536,0.709001
75%,0.736634,0.742928,0.635535,56.525555,55.500843,44.344610,0.723466,0.933277,0.566469,0.872380,0.643890,0.789110
max,0.887116,0.820128,0.821554,110.330559,78.147293,87.069519,0.899070,0.990066,0.837632,0.999071,0.820562,1.000000


### L6

In [41]:
mask = tta_ent_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.423954,0.612094,0.400691,33.999765,32.205593,28.468227,0.452228,0.810549,0.354731,0.491483,0.512856,0.645901
std,0.334599,0.179436,0.265551,34.221093,25.056279,26.201267,0.324628,0.192617,0.278857,0.409053,0.184871,0.250670
min,0.000000,0.146956,0.000000,4.123106,4.242640,4.000000,0.000000,0.146026,0.000000,0.000000,0.085574,0.000000
25%,0.037911,0.553460,0.157019,7.736254,8.870302,9.243416,0.094313,0.776723,0.088754,0.020135,0.399963,0.559666
50%,0.452372,0.675217,0.415055,19.209373,22.466639,16.881943,0.492290,0.880230,0.329232,0.613860,0.542903,0.708417
75%,0.734994,0.743693,0.634819,56.523478,55.513798,44.343399,0.719472,0.932797,0.571413,0.872381,0.642205,0.788189
max,0.886577,0.820068,0.820868,110.330559,78.185677,87.069519,0.899493,0.989781,0.838721,0.999330,0.820519,1.000000


In [42]:
mask = tta_ent_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.424600,0.611942,0.400684,33.969293,32.208800,28.468348,0.453074,0.810985,0.354168,0.491220,0.512525,0.646961
std,0.334966,0.179357,0.265551,34.242423,25.056023,26.201537,0.324889,0.192511,0.278452,0.409261,0.184869,0.250472
min,0.000000,0.147195,0.000000,4.123106,4.242640,4.000000,0.000000,0.146466,0.000000,0.000000,0.085520,0.000000
25%,0.037882,0.552858,0.157628,7.736254,8.870302,9.269696,0.098393,0.778030,0.089139,0.020119,0.398667,0.560837
50%,0.450174,0.674832,0.415527,19.131126,22.472204,16.881943,0.490885,0.880685,0.329559,0.613366,0.541308,0.708591
75%,0.736521,0.743027,0.635053,56.523478,55.534081,44.332783,0.721839,0.933310,0.567804,0.872817,0.642951,0.788599
max,0.886721,0.820027,0.821448,110.330559,78.115303,87.069519,0.899105,0.989876,0.838903,0.999358,0.820392,1.000000


In [43]:
mask = tta_ent_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_ent_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.425167,0.612132,0.400137,33.933304,32.230642,28.466473,0.452828,0.811172,0.353140,0.491214,0.512712,0.647597
std,0.335581,0.179398,0.265427,34.264506,25.084560,26.204569,0.325639,0.192529,0.278050,0.409405,0.184905,0.250159
min,0.000000,0.146999,0.000000,4.123106,4.242640,4.000000,0.000000,0.145911,0.000000,0.000000,0.085490,0.000000
25%,0.037915,0.553068,0.157990,7.702074,8.870302,9.243416,0.102062,0.777387,0.089360,0.020137,0.398363,0.562703
50%,0.451155,0.675000,0.415272,19.104973,22.449944,16.881943,0.490689,0.880852,0.329390,0.613106,0.541536,0.709001
75%,0.736634,0.742928,0.635535,56.525555,55.500843,44.344610,0.723466,0.933277,0.566469,0.872380,0.643890,0.789110
max,0.887116,0.820128,0.821554,110.330559,78.147293,87.069519,0.899070,0.990066,0.837632,0.999071,0.820562,1.000000


## With Supervision

In [44]:
tta_sup_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [45]:
tta_sup_lr4 = pd.read_csv(tta_sup_lr4_paths)
tta_sup_lr4 = clean_df(tta_sup_lr4, True, 'brats')
tta_sup_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.342515,0.532457,0.359910,32.159059,21.901205,32.626849,0.314722,0.559423,0.295363,0.602005,0.558922,0.628784
std,0.341506,0.262222,0.281427,30.042693,20.141020,30.859742,0.331474,0.294100,0.263200,0.442455,0.256927,0.331950
min,0.000000,0.000000,0.000000,3.741657,4.123106,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.330189,0.134293,9.110434,9.165152,9.433981,0.000000,0.336195,0.076083,0.000000,0.381946,0.516384
50%,0.212850,0.590553,0.332612,19.442223,13.416408,19.232780,0.154179,0.646313,0.233890,0.905873,0.619729,0.730177
75%,0.706434,0.751910,0.622812,51.148308,27.092434,50.849293,0.677217,0.792414,0.519573,0.979843,0.758545,0.884523
max,0.903202,0.861240,0.808336,105.016159,82.846237,110.914383,0.931949,0.976393,0.808431,1.000000,0.934519,1.000000


In [46]:
tta_sup_lr4_augs = pd.read_csv(tta_sup_lr4_augs_paths)
tta_sup_lr4_augs = clean_df(tta_sup_lr4_augs, True, 'brats')
tta_sup_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.220925,0.384316,0.485917,28.073108,28.083872,24.988776,0.151213,0.305498,0.440672,0.865766,0.671018,0.720054
std,0.247013,0.261179,0.279657,18.786958,13.616620,19.303722,0.188840,0.237495,0.284080,0.298665,0.287988,0.161673
min,0.001512,0.035095,0.002511,6.621739,11.000000,5.000000,0.000782,0.022150,0.001259,0.022487,0.084446,0.407694
25%,0.036780,0.177974,0.390388,11.354637,17.500000,7.995883,0.018757,0.110636,0.276584,0.930048,0.552670,0.679107
50%,0.109531,0.411338,0.551536,28.281096,26.500000,22.121397,0.058153,0.291815,0.500291,0.947144,0.796263,0.754750
75%,0.354057,0.608413,0.699992,39.034656,36.606064,32.804002,0.219235,0.459695,0.616573,0.994694,0.890644,0.804292
max,0.648944,0.699221,0.772236,66.475563,49.537865,56.348003,0.494111,0.661499,0.813053,1.000000,0.903638,0.910314


In [47]:
tta_sup_lr5 = pd.read_csv(tta_sup_lr5_paths)
tta_sup_lr5 = clean_df(tta_sup_lr5, True, 'brats')
tta_sup_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.367768,0.588418,0.369496,34.911323,27.164774,27.662696,0.385145,0.737623,0.305772,0.495294,0.507145,0.707042
std,0.314788,0.206841,0.281518,31.906189,22.369496,25.850898,0.319503,0.232373,0.269323,0.400598,0.198496,0.269777
min,0.000000,0.047013,0.000000,4.000000,4.000000,4.000000,0.000000,0.037823,0.000000,0.000000,0.062102,0.000000
25%,0.057966,0.457197,0.066685,7.816238,8.645485,9.716282,0.051161,0.630113,0.036235,0.030892,0.373358,0.631855
50%,0.381614,0.645627,0.389987,24.329389,17.918564,16.291562,0.363950,0.813417,0.261754,0.522232,0.499336,0.781433
75%,0.621393,0.751326,0.644747,51.696079,45.980145,36.732181,0.650532,0.913337,0.541568,0.871076,0.647584,0.887291
max,0.889603,0.897540,0.836687,103.639519,78.232338,93.661629,0.964245,0.989741,0.802792,1.000000,0.848252,1.000000


In [48]:
tta_sup_lr5_augs = pd.read_csv(tta_sup_lr5_augs_paths)
tta_sup_lr5_augs = clean_df(tta_sup_lr5_augs, True, 'brats')
tta_sup_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.251354,0.392505,0.353777,30.481213,35.882942,34.700411,0.206897,0.356656,0.287680,0.711581,0.634644,0.717450
std,0.217699,0.282857,0.281577,26.271068,26.350510,28.362993,0.240826,0.269079,0.255034,0.352650,0.306528,0.281410
min,0.000000,0.000000,0.000000,5.385165,5.099020,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044935,0.139972,0.044990,14.176036,18.791438,16.849542,0.023247,0.078220,0.023138,0.615960,0.582277,0.676033
50%,0.236568,0.394317,0.376739,23.692417,26.316006,25.217283,0.136575,0.422330,0.268174,0.875682,0.724266,0.768923
75%,0.360539,0.649398,0.617903,35.201440,47.299963,46.245535,0.249070,0.551522,0.530168,0.948627,0.878850,0.920083
max,0.630353,0.864868,0.750678,106.437775,98.009941,107.786827,0.819362,0.833447,0.685911,1.000000,0.920550,1.000000


In [49]:
tta_sup_lr6 = pd.read_csv(tta_sup_lr6_paths)
tta_sup_lr6 = clean_df(tta_sup_lr6, True, 'brats')
tta_sup_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.366634,0.571329,0.368123,35.586927,29.986062,29.858129,0.414482,0.769006,0.312282,0.448508,0.474599,0.667942
std,0.306594,0.196153,0.285014,31.444269,22.990595,26.007447,0.315751,0.219446,0.276910,0.388745,0.189606,0.281504
min,0.000000,0.136262,0.000000,4.000000,3.741657,4.000000,0.000000,0.126507,0.000000,0.000000,0.083178,0.000000
25%,0.032131,0.447419,0.062258,9.695360,10.488089,8.774964,0.029564,0.651897,0.032191,0.028594,0.349704,0.566553
50%,0.386949,0.602569,0.371576,26.305893,20.223749,18.357559,0.456115,0.845921,0.245130,0.378698,0.444696,0.731334
75%,0.619724,0.716474,0.621666,53.047150,48.093658,47.686478,0.687662,0.936704,0.548624,0.819261,0.605177,0.882150
max,0.888434,0.891640,0.845145,107.083977,78.351151,92.643127,0.965398,0.989312,0.831729,1.000000,0.830449,1.000000


In [50]:
tta_sup_lr6_augs = pd.read_csv(tta_sup_lr6_augs_paths)
tta_sup_lr6_augs = clean_df(tta_sup_lr6_augs, True, 'brats')
tta_sup_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.272153,0.407633,0.362298,25.757366,31.257121,31.939110,0.233620,0.401815,0.296752,0.699753,0.644629,0.741389
std,0.238035,0.275667,0.291305,19.944386,21.621353,27.902705,0.258186,0.291890,0.264273,0.302620,0.261762,0.266192
min,0.000000,0.000000,0.000000,5.656854,5.099020,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.037532,0.158210,0.027724,12.464551,17.355343,15.393839,0.019543,0.089137,0.014094,0.491920,0.514296,0.669362
50%,0.280158,0.451704,0.362912,22.308077,25.573423,23.108440,0.167284,0.456651,0.256664,0.839449,0.749117,0.784131
75%,0.484951,0.643368,0.624376,30.479127,35.319727,30.647572,0.392392,0.614664,0.532377,0.917712,0.841099,0.921386
max,0.655453,0.866928,0.782154,79.172913,89.553612,107.029198,0.871901,0.852970,0.786232,1.000000,0.884899,1.000000


In [169]:
tta_sup_first_layer_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_first_layer_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_first_layer_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_first_layer_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_first_layer_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_first_layer_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_first_layer_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_first_layer_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_first_layer_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_first_layer_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_first_layer_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [170]:
tta_sup_first_layer_lr4 = pd.read_csv(tta_sup_first_layer_lr4_paths)
tta_sup_first_layer_lr4 = clean_df(tta_sup_first_layer_lr4, True, 'brats')
tta_sup_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.331557,0.548852,0.389012,32.549683,20.203066,33.303043,0.314728,0.579259,0.310786,0.597439,0.569733,0.644429
std,0.350955,0.262830,0.270594,29.264011,19.394570,33.852890,0.350503,0.296120,0.243962,0.438186,0.255785,0.319276
min,0.000000,0.000000,0.000000,3.548985,4.123106,3.741657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.343629,0.200785,8.062258,8.602325,8.497008,0.000000,0.377285,0.111669,0.000000,0.386342,0.538276
50%,0.159653,0.630792,0.406633,23.021729,13.304134,16.881943,0.100454,0.675323,0.281609,0.872340,0.618843,0.737225
75%,0.728675,0.752500,0.613260,51.623638,22.226110,51.720402,0.673474,0.841130,0.496718,0.967116,0.748367,0.874509
max,0.910428,0.907000,0.820618,100.667770,83.971420,109.099953,0.932447,0.988724,0.780064,1.000000,0.948117,1.000000


In [53]:
tta_sup_first_layer_lr4_augs = pd.read_csv(tta_sup_first_layer_lr4_augs_paths)
tta_sup_first_layer_lr4_augs = clean_df(tta_sup_first_layer_lr4_augs, True, 'brats')
tta_sup_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.239557,0.379174,0.391884,26.720631,34.113359,26.597019,0.165535,0.299002,0.332257,0.896609,0.692584,0.783233
std,0.259795,0.246807,0.260045,15.179190,16.602129,16.227330,0.194913,0.224090,0.266918,0.192585,0.243257,0.166463
min,0.001697,0.015010,0.022434,6.621739,14.000000,4.000000,0.000849,0.007626,0.011365,0.423514,0.198586,0.385662
25%,0.031455,0.172945,0.178456,15.823272,19.000000,14.421460,0.016203,0.116669,0.101730,0.928640,0.527951,0.693605
50%,0.087948,0.387351,0.352749,26.000000,32.264530,26.410078,0.045997,0.247839,0.227811,0.969466,0.812137,0.833542
75%,0.501708,0.633434,0.627936,34.930719,48.324589,34.015379,0.345596,0.505839,0.542422,0.994195,0.876694,0.912828
max,0.665782,0.681144,0.745985,57.442139,60.000000,56.320946,0.510071,0.694179,0.728598,1.000000,0.940856,0.987701


In [54]:
tta_sup_first_layer_lr5 = pd.read_csv(tta_sup_first_layer_lr5_paths)
tta_sup_first_layer_lr5 = clean_df(tta_sup_first_layer_lr5, True, 'brats')
tta_sup_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.367768,0.588418,0.369496,34.911323,27.164774,27.662696,0.385145,0.737623,0.305772,0.495294,0.507145,0.707042
std,0.314788,0.206841,0.281518,31.906189,22.369496,25.850898,0.319503,0.232373,0.269323,0.400598,0.198496,0.269777
min,0.000000,0.047013,0.000000,4.000000,4.000000,4.000000,0.000000,0.037823,0.000000,0.000000,0.062102,0.000000
25%,0.057966,0.457197,0.066685,7.816238,8.645485,9.716282,0.051161,0.630113,0.036235,0.030892,0.373358,0.631855
50%,0.381614,0.645627,0.389987,24.329389,17.918564,16.291562,0.363950,0.813417,0.261754,0.522232,0.499336,0.781433
75%,0.621393,0.751326,0.644747,51.696079,45.980145,36.732181,0.650532,0.913337,0.541568,0.871076,0.647584,0.887291
max,0.889603,0.897540,0.836687,103.639519,78.232338,93.661629,0.964245,0.989741,0.802792,1.000000,0.848252,1.000000


In [55]:
tta_sup_first_layer_lr5 = pd.read_csv(tta_sup_first_layer_lr5_paths)
tta_sup_first_layer_lr5 = clean_df(tta_sup_first_layer_lr5, True, 'brats')
tta_sup_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.367768,0.588418,0.369496,34.911323,27.164774,27.662696,0.385145,0.737623,0.305772,0.495294,0.507145,0.707042
std,0.314788,0.206841,0.281518,31.906189,22.369496,25.850898,0.319503,0.232373,0.269323,0.400598,0.198496,0.269777
min,0.000000,0.047013,0.000000,4.000000,4.000000,4.000000,0.000000,0.037823,0.000000,0.000000,0.062102,0.000000
25%,0.057966,0.457197,0.066685,7.816238,8.645485,9.716282,0.051161,0.630113,0.036235,0.030892,0.373358,0.631855
50%,0.381614,0.645627,0.389987,24.329389,17.918564,16.291562,0.363950,0.813417,0.261754,0.522232,0.499336,0.781433
75%,0.621393,0.751326,0.644747,51.696079,45.980145,36.732181,0.650532,0.913337,0.541568,0.871076,0.647584,0.887291
max,0.889603,0.897540,0.836687,103.639519,78.232338,93.661629,0.964245,0.989741,0.802792,1.000000,0.848252,1.000000


In [56]:
tta_sup_first_layer_lr5_augs = pd.read_csv(tta_sup_first_layer_lr5_augs_paths)
tta_sup_first_layer_lr5_augs = clean_df(tta_sup_first_layer_lr5_augs, True, 'brats')
tta_sup_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.258726,0.413285,0.362976,28.586808,35.724924,34.472690,0.215567,0.375473,0.299839,0.755115,0.614487,0.704239
std,0.228049,0.279342,0.286691,23.763291,27.279023,29.627305,0.248338,0.260677,0.265294,0.319105,0.305912,0.283605
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.046814,0.151965,0.071004,15.390553,17.632169,14.973489,0.024113,0.115314,0.037276,0.715476,0.546670,0.645919
50%,0.246644,0.499649,0.395114,23.769728,25.258661,28.792360,0.141981,0.473573,0.271047,0.903317,0.704671,0.772047
75%,0.410292,0.650136,0.652220,32.394442,47.892071,43.808707,0.320126,0.553226,0.550119,0.957427,0.859824,0.886351
max,0.631386,0.862915,0.741663,97.348854,98.101471,107.786827,0.824085,0.829786,0.740913,1.000000,0.915540,1.000000


In [57]:
tta_sup_first_layer_lr6 = pd.read_csv(tta_sup_first_layer_lr6_paths)
tta_sup_first_layer_lr6 = clean_df(tta_sup_first_layer_lr6, True, 'brats')
tta_sup_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.368625,0.588740,0.373956,34.829342,27.173170,27.506155,0.381851,0.740263,0.310111,0.504349,0.506253,0.705435
std,0.316500,0.206668,0.282846,31.813700,22.409378,25.670139,0.320563,0.233695,0.271435,0.405596,0.197298,0.266530
min,0.000000,0.051695,0.000000,4.123106,4.000000,4.000000,0.000000,0.041748,0.000000,0.000000,0.067864,0.000000
25%,0.057954,0.455357,0.073403,8.278015,8.774964,9.427127,0.052500,0.625908,0.038984,0.030854,0.380805,0.648506
50%,0.377780,0.649957,0.396337,24.266356,17.091625,16.309046,0.355490,0.818372,0.277280,0.536103,0.503480,0.779604
75%,0.635988,0.753761,0.646097,52.389900,45.966988,36.052417,0.627078,0.916139,0.557583,0.907727,0.644322,0.876829
max,0.888607,0.895226,0.836523,103.639519,78.262383,93.661629,0.964245,0.990487,0.808532,1.000000,0.842695,1.000000


In [58]:
tta_sup_first_layer_lr6_augs = pd.read_csv(tta_sup_first_layer_lr6_augs_paths)
tta_sup_first_layer_lr6_augs = clean_df(tta_sup_first_layer_lr6_augs, True, 'brats')
tta_sup_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.274043,0.408696,0.361812,25.606843,31.257813,31.934914,0.235283,0.402499,0.295901,0.700565,0.644781,0.741983
std,0.239420,0.274997,0.290934,19.972092,21.622248,27.911878,0.258928,0.291120,0.263503,0.302853,0.261737,0.266092
min,0.000000,0.000000,0.000000,5.656854,5.099020,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.037853,0.157738,0.028220,12.464551,17.339387,15.393839,0.019719,0.088765,0.014348,0.493195,0.514545,0.669825
50%,0.282706,0.451762,0.362040,22.226110,25.573423,23.108440,0.167913,0.456474,0.255247,0.839572,0.747952,0.784612
75%,0.484676,0.643280,0.624228,30.494806,35.318684,30.565856,0.394163,0.614756,0.530887,0.917604,0.840789,0.920634
max,0.663014,0.866695,0.781767,79.183334,89.553612,107.029198,0.872491,0.852379,0.784129,1.000000,0.884661,1.000000


In [59]:
tta_sup_norm_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_norm_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_norm_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_norm_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_norm_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_norm_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_norm_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_norm_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_norm_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_norm_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_norm_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_norm_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [60]:
tta_sup_norm_lr4 = pd.read_csv(tta_sup_norm_lr4_paths)
tta_sup_norm_lr4 = clean_df(tta_sup_norm_lr4, True, 'brats')
tta_sup_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.343952,0.555932,0.376677,30.590991,21.341626,34.236796,0.321328,0.588863,0.304192,0.579985,0.585858,0.640414
std,0.350354,0.248438,0.275275,26.454735,20.470302,33.742105,0.342541,0.281850,0.251086,0.437346,0.245534,0.317257
min,0.000000,0.000000,0.000000,4.000000,4.123106,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.352185,0.161270,8.093029,8.638090,9.923840,0.000000,0.386593,0.090094,0.000000,0.401284,0.547914
50%,0.197796,0.638580,0.379393,22.010897,14.340859,17.262087,0.138585,0.660507,0.266088,0.840348,0.625756,0.729724
75%,0.740115,0.749078,0.609352,50.055931,23.307542,52.810436,0.683690,0.817185,0.526344,0.971240,0.755857,0.869574
max,0.909591,0.906619,0.822310,99.391151,82.846237,110.607414,0.925029,0.976448,0.737554,1.000000,0.941965,1.000000


In [61]:
tta_sup_norm_lr4_augs = pd.read_csv(tta_sup_norm_lr4_augs_paths)
tta_sup_norm_lr4_augs = clean_df(tta_sup_norm_lr4_augs, True, 'brats')
tta_sup_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000,12.000000
mean,0.213762,0.359233,0.410744,29.074524,34.462378,27.684084,0.146705,0.284903,0.362832,0.915230,0.672335,0.751453
std,0.244285,0.251777,0.287020,16.989601,17.969120,19.202504,0.186944,0.234031,0.287934,0.171958,0.259280,0.184986
min,0.000485,0.051620,0.016367,6.164414,10.488089,4.898980,0.000243,0.027259,0.008334,0.385515,0.134581,0.405794
25%,0.036270,0.149543,0.113106,14.671329,19.750000,9.500000,0.019078,0.082411,0.060323,0.917513,0.522636,0.684038
50%,0.119087,0.267570,0.510513,27.693064,30.000000,27.213939,0.063333,0.190236,0.423044,0.974182,0.807127,0.752042
75%,0.311911,0.625701,0.669320,41.416101,49.430417,39.515445,0.201790,0.485048,0.583471,0.996720,0.867864,0.913342
max,0.629153,0.683655,0.744596,59.000000,60.008331,56.329388,0.468302,0.665875,0.784335,1.000000,0.913092,0.983326


In [62]:
tta_sup_norm_lr5 = pd.read_csv(tta_sup_norm_lr5_paths)
tta_sup_norm_lr5 = clean_df(tta_sup_norm_lr5, True, 'brats')
tta_sup_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.367665,0.588656,0.373863,35.002715,27.210568,27.462522,0.382221,0.739670,0.310586,0.501169,0.506577,0.708418
std,0.315446,0.206526,0.282592,31.947259,22.394546,25.890945,0.318846,0.232588,0.271544,0.406387,0.197685,0.268977
min,0.000000,0.050147,0.000000,4.123106,4.000000,4.000000,0.000000,0.040122,0.000000,0.000000,0.066849,0.000000
25%,0.057377,0.455815,0.070173,8.278015,8.688932,9.467683,0.052476,0.625428,0.037498,0.030728,0.377197,0.642985
50%,0.367073,0.650485,0.396180,24.450212,17.972115,15.413600,0.359698,0.816788,0.277107,0.529935,0.502257,0.778392
75%,0.630475,0.750664,0.645178,52.390532,45.930573,35.676996,0.632235,0.915227,0.554397,0.905370,0.648113,0.879584
max,0.888466,0.895769,0.835904,103.739037,78.287933,93.661629,0.964245,0.989638,0.810840,1.000000,0.844030,1.000000


In [63]:
tta_sup_norm_lr5_augs = pd.read_csv(tta_sup_norm_lr5_augs_paths)
tta_sup_norm_lr5_augs = clean_df(tta_sup_norm_lr5_augs, True, 'brats')
tta_sup_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.232649,0.383600,0.356360,28.771352,35.194667,33.823872,0.193708,0.347522,0.288953,0.725327,0.621856,0.714752
std,0.224576,0.282528,0.285497,23.310918,25.854152,28.032935,0.242695,0.267778,0.253437,0.345414,0.295828,0.271333
min,0.000000,0.000000,0.000000,5.000000,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.037689,0.141160,0.059848,17.262676,18.000000,17.000000,0.019283,0.079770,0.031123,0.712815,0.491172,0.692035
50%,0.206129,0.300118,0.396696,24.020824,27.013885,28.017851,0.115004,0.392022,0.267869,0.878173,0.711112,0.768240
75%,0.306645,0.652713,0.633572,32.155869,46.765373,31.644108,0.195339,0.553223,0.514361,0.939611,0.866063,0.868549
max,0.627962,0.862635,0.741728,97.348854,98.101471,107.786827,0.809327,0.830043,0.666099,1.000000,0.913480,1.000000


In [64]:
tta_sup_norm_lr6 = pd.read_csv(tta_sup_norm_lr6_paths)
tta_sup_norm_lr6 = clean_df(tta_sup_norm_lr6, True, 'brats')
tta_sup_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.367078,0.571412,0.368122,35.585185,29.997133,29.865046,0.414609,0.768930,0.312139,0.448791,0.474738,0.668579
std,0.306875,0.196149,0.285006,31.447576,22.986646,26.029159,0.315712,0.219521,0.276884,0.388923,0.189568,0.280898
min,0.000000,0.136419,0.000000,4.000000,3.741657,4.000000,0.000000,0.126437,0.000000,0.000000,0.082931,0.000000
25%,0.032685,0.446920,0.062258,9.695360,10.488089,8.774964,0.030083,0.652619,0.032191,0.028591,0.349729,0.567872
50%,0.387521,0.603336,0.371902,26.305893,20.223749,18.357559,0.458283,0.845796,0.245193,0.379848,0.444574,0.731513
75%,0.621438,0.716741,0.621597,53.038193,48.093658,47.686478,0.685810,0.936589,0.546817,0.819242,0.604788,0.882618
max,0.888582,0.891552,0.845170,106.898552,78.364532,92.643127,0.965398,0.989233,0.831820,1.000000,0.830218,1.000000


In [65]:
tta_sup_norm_lr6_augs = pd.read_csv(tta_sup_norm_lr6_augs_paths)
tta_sup_norm_lr6_augs = clean_df(tta_sup_norm_lr6_augs, True, 'brats')
tta_sup_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.273364,0.407771,0.362028,25.720339,31.253846,32.340417,0.234404,0.401893,0.296480,0.700537,0.644955,0.741453
std,0.238494,0.275641,0.291487,19.933268,21.622988,27.937703,0.258065,0.291905,0.264128,0.302857,0.261745,0.266185
min,0.000000,0.000000,0.000000,5.656854,5.099020,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.037593,0.158728,0.027792,12.464551,17.323399,15.340739,0.019578,0.089224,0.014129,0.492117,0.514932,0.669104
50%,0.281922,0.452448,0.363042,22.037474,25.573423,23.108440,0.167354,0.456480,0.256799,0.839965,0.753547,0.783908
75%,0.484867,0.643402,0.624382,30.494806,35.318684,34.767221,0.391348,0.614575,0.532314,0.918150,0.840802,0.921474
max,0.656875,0.866664,0.782179,79.183334,89.553612,107.029198,0.870720,0.852277,0.785785,1.000000,0.885114,1.000000


In [66]:
tta_sup_decoder_lr4_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_decoder_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_decoder_lr5_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_decoder_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_sup_decoder_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_decoder_lr6_paths =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_sup_decoder_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_sup_decoder_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [67]:
tta_sup_decoder_lr6_augs = pd.read_csv(tta_sup_decoder_lr6_augs_paths)
tta_sup_decoder_lr6_augs = clean_df(tta_sup_decoder_lr6_augs, True, 'brats')
tta_sup_decoder_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.272153,0.407633,0.362298,25.757366,31.257121,31.939110,0.233620,0.401815,0.296752,0.699753,0.644629,0.741389
std,0.238035,0.275667,0.291305,19.944386,21.621353,27.902705,0.258186,0.291890,0.264273,0.302620,0.261762,0.266192
min,0.000000,0.000000,0.000000,5.656854,5.099020,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.037532,0.158210,0.027724,12.464551,17.355343,15.393839,0.019543,0.089137,0.014094,0.491920,0.514296,0.669362
50%,0.280158,0.451704,0.362912,22.308077,25.573423,23.108440,0.167284,0.456651,0.256664,0.839449,0.749117,0.784131
75%,0.484951,0.643368,0.624376,30.479127,35.319727,30.647572,0.392392,0.614664,0.532377,0.917712,0.841099,0.921386
max,0.655453,0.866928,0.782154,79.172913,89.553612,107.029198,0.871901,0.852970,0.786232,1.000000,0.884899,1.000000


In [68]:
tta_sup_decoder_lr6 = pd.read_csv(tta_sup_decoder_lr6_paths)
tta_sup_decoder_lr6 = clean_df(tta_sup_decoder_lr6, True, 'brats')
tta_sup_decoder_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.366634,0.571329,0.368123,35.586927,29.986062,29.858129,0.414482,0.769006,0.312282,0.448508,0.474599,0.667942
std,0.306594,0.196153,0.285014,31.444269,22.990595,26.007447,0.315751,0.219446,0.276910,0.388745,0.189606,0.281504
min,0.000000,0.136262,0.000000,4.000000,3.741657,4.000000,0.000000,0.126507,0.000000,0.000000,0.083178,0.000000
25%,0.032131,0.447419,0.062258,9.695360,10.488089,8.774964,0.029564,0.651897,0.032191,0.028594,0.349704,0.566553
50%,0.386949,0.602569,0.371576,26.305893,20.223749,18.357559,0.456115,0.845921,0.245130,0.378698,0.444696,0.731334
75%,0.619724,0.716474,0.621666,53.047150,48.093658,47.686478,0.687662,0.936704,0.548624,0.819261,0.605177,0.882150
max,0.888434,0.891640,0.845145,107.083977,78.351151,92.643127,0.965398,0.989312,0.831729,1.000000,0.830449,1.000000


In [69]:
tta_sup_decoder_lr5_augs = pd.read_csv(tta_sup_decoder_lr5_augs_paths)
tta_sup_decoder_lr5_augs = clean_df(tta_sup_decoder_lr5_augs, True, 'brats')
tta_sup_decoder_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.251354,0.392505,0.353777,30.481213,35.882942,34.700411,0.206897,0.356656,0.287680,0.711581,0.634644,0.717450
std,0.217699,0.282857,0.281577,26.271068,26.350510,28.362993,0.240826,0.269079,0.255034,0.352650,0.306528,0.281410
min,0.000000,0.000000,0.000000,5.385165,5.099020,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044935,0.139972,0.044990,14.176036,18.791438,16.849542,0.023247,0.078220,0.023138,0.615960,0.582277,0.676033
50%,0.236568,0.394317,0.376739,23.692417,26.316006,25.217283,0.136575,0.422330,0.268174,0.875682,0.724266,0.768923
75%,0.360539,0.649398,0.617903,35.201440,47.299963,46.245535,0.249070,0.551522,0.530168,0.948627,0.878850,0.920083
max,0.630353,0.864868,0.750678,106.437775,98.009941,107.786827,0.819362,0.833447,0.685911,1.000000,0.920550,1.000000


In [70]:
tta_sup_decoder_lr5 = pd.read_csv(tta_sup_decoder_lr5_paths)
tta_sup_decoder_lr5 = clean_df(tta_sup_decoder_lr5, True, 'brats')
tta_sup_decoder_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.367768,0.588418,0.369496,34.911323,27.164774,27.662696,0.385145,0.737623,0.305772,0.495294,0.507145,0.707042
std,0.314788,0.206841,0.281518,31.906189,22.369496,25.850898,0.319503,0.232373,0.269323,0.400598,0.198496,0.269777
min,0.000000,0.047013,0.000000,4.000000,4.000000,4.000000,0.000000,0.037823,0.000000,0.000000,0.062102,0.000000
25%,0.057966,0.457197,0.066685,7.816238,8.645485,9.716282,0.051161,0.630113,0.036235,0.030892,0.373358,0.631855
50%,0.381614,0.645627,0.389987,24.329389,17.918564,16.291562,0.363950,0.813417,0.261754,0.522232,0.499336,0.781433
75%,0.621393,0.751326,0.644747,51.696079,45.980145,36.732181,0.650532,0.913337,0.541568,0.871076,0.647584,0.887291
max,0.889603,0.897540,0.836687,103.639519,78.232338,93.661629,0.964245,0.989741,0.802792,1.000000,0.848252,1.000000


In [71]:
tta_sup_decoder_lr4 = pd.read_csv(tta_sup_decoder_lr4_paths)
tta_sup_decoder_lr4 = clean_df(tta_sup_decoder_lr4, True, 'brats')
tta_sup_decoder_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.342515,0.532457,0.359910,32.159059,21.901205,32.626849,0.314722,0.559423,0.295363,0.602005,0.558922,0.628784
std,0.341506,0.262222,0.281427,30.042693,20.141020,30.859742,0.331474,0.294100,0.263200,0.442455,0.256927,0.331950
min,0.000000,0.000000,0.000000,3.741657,4.123106,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.330189,0.134293,9.110434,9.165152,9.433981,0.000000,0.336195,0.076083,0.000000,0.381946,0.516384
50%,0.212850,0.590553,0.332612,19.442223,13.416408,19.232780,0.154179,0.646313,0.233890,0.905873,0.619729,0.730177
75%,0.706434,0.751910,0.622812,51.148308,27.092434,50.849293,0.677217,0.792414,0.519573,0.979843,0.758545,0.884523
max,0.903202,0.861240,0.808336,105.016159,82.846237,110.914383,0.931949,0.976393,0.808431,1.000000,0.934519,1.000000


In [72]:
tta_sup_decoder_lr4_augs = pd.read_csv(tta_sup_decoder_lr4_augs_paths)
tta_sup_decoder_lr4_augs = clean_df(tta_sup_decoder_lr4_augs, True, 'brats')
tta_sup_decoder_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000,10.000000
mean,0.220925,0.384316,0.485917,28.073108,28.083872,24.988776,0.151213,0.305498,0.440672,0.865766,0.671018,0.720054
std,0.247013,0.261179,0.279657,18.786958,13.616620,19.303722,0.188840,0.237495,0.284080,0.298665,0.287988,0.161673
min,0.001512,0.035095,0.002511,6.621739,11.000000,5.000000,0.000782,0.022150,0.001259,0.022487,0.084446,0.407694
25%,0.036780,0.177974,0.390388,11.354637,17.500000,7.995883,0.018757,0.110636,0.276584,0.930048,0.552670,0.679107
50%,0.109531,0.411338,0.551536,28.281096,26.500000,22.121397,0.058153,0.291815,0.500291,0.947144,0.796263,0.754750
75%,0.354057,0.608413,0.699992,39.034656,36.606064,32.804002,0.219235,0.459695,0.616573,0.994694,0.890644,0.804292
max,0.648944,0.699221,0.772236,66.475563,49.537865,56.348003,0.494111,0.661499,0.813053,1.000000,0.903638,0.910314


### LR 4

In [73]:
mask = tta_sup_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_sup_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.445841,0.597052,0.370282,28.367747,20.424521,28.820877,0.390164,0.640209,0.307120,0.659311,0.611924,0.686811
std,0.359125,0.240142,0.273996,28.655763,19.045084,27.086900,0.340501,0.267504,0.268387,0.412948,0.240095,0.283577
min,0.000000,0.002413,0.000000,3.741657,4.242640,4.000000,0.000000,0.002126,0.000000,0.000000,0.002790,0.000000
25%,0.056277,0.558108,0.149238,7.810250,8.774964,7.348469,0.029731,0.534120,0.081938,0.250000,0.578952,0.609302
50%,0.508893,0.690883,0.332612,11.180340,13.416408,18.110771,0.386605,0.742254,0.233890,0.906343,0.656932,0.780354
75%,0.810651,0.754980,0.622812,40.816051,20.322401,44.821869,0.743769,0.816394,0.519573,0.968217,0.768408,0.814554
max,0.903202,0.861240,0.808336,105.016159,79.305740,87.662430,0.868412,0.976393,0.808431,0.998195,0.934519,1.000000


In [74]:
mask = tta_sup_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_sup_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.398813,0.529935,0.418041,34.887367,40.214312,26.207798,0.451349,0.844692,0.363435,0.524569,0.409572,0.674249
std,0.327574,0.167569,0.264667,26.787275,21.951498,27.926907,0.341906,0.173149,0.270350,0.418283,0.176611,0.238871
min,0.000000,0.119377,0.000000,4.000000,7.549834,5.000000,0.000000,0.187761,0.000000,0.000000,0.065714,0.000000
25%,0.019171,0.434392,0.247709,9.899495,19.723083,7.280110,0.009678,0.804496,0.161547,0.046972,0.286842,0.562364
50%,0.403662,0.494986,0.396193,29.017237,41.424629,13.453624,0.495812,0.915466,0.363625,0.638380,0.359304,0.725324
75%,0.752336,0.657340,0.651400,52.354561,59.539902,39.440445,0.764076,0.955399,0.591151,0.968087,0.502047,0.787957
max,0.856765,0.816531,0.852126,103.670151,78.192070,113.021019,0.914928,0.992901,0.828995,1.000000,0.817408,0.990136


In [75]:
mask = tta_sup_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.439361,0.605964,0.374822,27.746823,21.723295,31.362939,0.391617,0.657913,0.299177,0.629664,0.623663,0.684667
std,0.371520,0.230038,0.261860,26.573076,20.619206,31.757482,0.355150,0.260066,0.241578,0.416076,0.235903,0.276378
min,0.000000,0.005124,0.000000,4.000000,4.242640,4.000000,0.000000,0.004559,0.000000,0.000000,0.005851,0.000000
25%,0.000000,0.592224,0.170232,7.648459,8.485281,10.000000,0.000000,0.593900,0.097851,0.000000,0.577775,0.578725
50%,0.567573,0.679307,0.354299,15.684387,14.224886,15.524175,0.484582,0.735669,0.247307,0.863198,0.669306,0.766482
75%,0.818566,0.760336,0.599101,40.545654,26.551836,43.661205,0.740062,0.836396,0.523674,0.961504,0.759850,0.840015
max,0.909591,0.865236,0.803813,99.391151,79.469490,110.059074,0.920033,0.976448,0.716906,0.996755,0.941965,1.000000


### L5

In [76]:
mask = tta_sup_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_sup_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.424580,0.625956,0.389427,35.201649,29.743414,28.969003,0.428241,0.783512,0.332097,0.510060,0.540128,0.690869
std,0.342426,0.180822,0.269926,34.066437,23.800243,27.904109,0.329876,0.201294,0.271681,0.413372,0.180731,0.241712
min,0.000000,0.128133,0.000000,4.000000,4.898980,4.000000,0.000000,0.123367,0.000000,0.000000,0.087113,0.000000
25%,0.065353,0.567250,0.156828,7.071068,8.602325,9.505208,0.071901,0.725753,0.092129,0.037843,0.447153,0.626593
50%,0.472473,0.686586,0.414731,21.563858,17.920654,15.066519,0.506300,0.835785,0.326420,0.637264,0.579437,0.766787
75%,0.711226,0.752009,0.642704,52.125328,50.447994,43.307606,0.716981,0.913558,0.559737,0.911824,0.647693,0.829106
max,0.889603,0.825966,0.836687,103.639519,78.232338,89.117615,0.893389,0.989741,0.802792,0.995932,0.834071,1.000000


In [77]:
mask = tta_sup_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_sup_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.424580,0.625956,0.389427,35.201649,29.743414,28.969003,0.428241,0.783512,0.332097,0.510060,0.540128,0.690869
std,0.342426,0.180822,0.269926,34.066437,23.800243,27.904109,0.329876,0.201294,0.271681,0.413372,0.180731,0.241712
min,0.000000,0.128133,0.000000,4.000000,4.898980,4.000000,0.000000,0.123367,0.000000,0.000000,0.087113,0.000000
25%,0.065353,0.567250,0.156828,7.071068,8.602325,9.505208,0.071901,0.725753,0.092129,0.037843,0.447153,0.626593
50%,0.472473,0.686586,0.414731,21.563858,17.920654,15.066519,0.506300,0.835785,0.326420,0.637264,0.579437,0.766787
75%,0.711226,0.752009,0.642704,52.125328,50.447994,43.307606,0.716981,0.913558,0.559737,0.911824,0.647693,0.829106
max,0.889603,0.825966,0.836687,103.639519,78.232338,89.117615,0.893389,0.989741,0.802792,0.995932,0.834071,1.000000


In [78]:
mask = tta_sup_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.426283,0.626408,0.392608,35.256301,29.796248,29.010875,0.427679,0.785951,0.335429,0.511582,0.539809,0.692183
std,0.344222,0.180944,0.269751,34.190401,23.839360,27.919111,0.330217,0.201178,0.272747,0.415218,0.180598,0.240972
min,0.000000,0.124140,0.000000,4.123106,4.582576,4.000000,0.000000,0.118847,0.000000,0.000000,0.085730,0.000000
25%,0.062939,0.567680,0.164858,7.071068,8.660254,8.774964,0.069805,0.727652,0.098030,0.037160,0.447123,0.634469
50%,0.483990,0.686386,0.419045,21.656408,18.027756,14.142136,0.505305,0.839401,0.330176,0.619275,0.582767,0.764179
75%,0.721028,0.750695,0.641993,52.493809,50.361198,43.000000,0.725557,0.915337,0.560175,0.915062,0.650165,0.843793
max,0.888466,0.826199,0.835904,103.739037,78.287933,89.117615,0.892232,0.989638,0.810840,0.994033,0.832540,1.000000


In [79]:
mask = tta_sup_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.264584,0.410808,0.371389,29.029681,35.718887,34.263590,0.217786,0.374221,0.302313,0.749032,0.619596,0.704797
std,0.215246,0.278174,0.277744,26.154002,27.062080,29.071005,0.242313,0.264410,0.253250,0.318842,0.307244,0.283215
min,0.000000,0.000000,0.000000,5.385165,5.099020,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.091058,0.157216,0.122735,13.587248,17.582875,16.699083,0.048213,0.105576,0.067916,0.684723,0.547256,0.664155
50%,0.257423,0.507946,0.406702,23.280893,25.632011,21.921412,0.150936,0.448124,0.270249,0.877219,0.714245,0.763952
75%,0.406100,0.651971,0.628750,32.836348,47.866642,43.786639,0.300575,0.555684,0.534514,0.959064,0.858389,0.901321
max,0.630353,0.864868,0.750678,106.437775,98.009941,107.786827,0.819362,0.833447,0.685911,1.000000,0.914499,1.000000


In [80]:
mask = tta_sup_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.258726,0.413285,0.362976,28.586808,35.724924,34.472690,0.215567,0.375473,0.299839,0.755115,0.614487,0.704239
std,0.228049,0.279342,0.286691,23.763291,27.279023,29.627305,0.248338,0.260677,0.265294,0.319105,0.305912,0.283605
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.046814,0.151965,0.071004,15.390553,17.632169,14.973489,0.024113,0.115314,0.037276,0.715476,0.546670,0.645919
50%,0.246644,0.499649,0.395114,23.769728,25.258661,28.792360,0.141981,0.473573,0.271047,0.903317,0.704671,0.772047
75%,0.410292,0.650136,0.652220,32.394442,47.892071,43.808707,0.320126,0.553226,0.550119,0.957427,0.859824,0.886351
max,0.631386,0.862915,0.741663,97.348854,98.101471,107.786827,0.824085,0.829786,0.740913,1.000000,0.915540,1.000000


In [81]:
mask = tta_sup_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.256977,0.408566,0.359007,29.039381,35.628681,34.909656,0.214017,0.373832,0.292203,0.749045,0.615881,0.704692
std,0.222438,0.283281,0.283425,24.537165,27.181892,29.288841,0.246726,0.266902,0.254301,0.313934,0.304858,0.283554
min,0.000000,0.000000,0.000000,5.000000,5.000000,4.690416,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.049051,0.156809,0.060281,14.836175,16.632169,16.954287,0.025276,0.105565,0.031289,0.744149,0.546094,0.661896
50%,0.253237,0.505417,0.396696,24.020824,25.632011,28.930952,0.146338,0.448432,0.267869,0.878173,0.711112,0.768240
75%,0.403322,0.655587,0.627783,32.984285,47.887789,43.813124,0.304121,0.558643,0.527341,0.932353,0.857423,0.894792
max,0.627962,0.862635,0.741728,97.348854,98.101471,107.786827,0.809327,0.830043,0.666099,1.000000,0.913480,1.000000


### L6

In [82]:
mask = tta_sup_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.413092,0.607620,0.383889,35.023918,31.898530,30.733581,0.439255,0.803954,0.338616,0.473084,0.508307,0.643565
std,0.339024,0.174891,0.273205,34.172347,24.540062,27.676413,0.330067,0.192423,0.280770,0.409329,0.178234,0.251011
min,0.000000,0.147677,0.000000,4.000000,4.242640,4.000000,0.000000,0.170293,0.000000,0.000000,0.083178,0.000000
25%,0.000000,0.541804,0.100775,8.229187,9.091215,8.751316,0.000000,0.729062,0.055246,0.000000,0.409811,0.566206
50%,0.456656,0.663461,0.395378,19.762367,22.610153,20.447582,0.483504,0.876767,0.291889,0.501941,0.542368,0.702711
75%,0.702992,0.730804,0.625870,55.344021,51.136482,50.735952,0.702620,0.937143,0.559827,0.868221,0.619145,0.785002
max,0.888434,0.815590,0.816965,107.083977,78.351151,86.205566,0.902594,0.989312,0.831729,1.000000,0.808414,1.000000


In [83]:
mask = tta_sup_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.429405,0.626830,0.393301,35.140283,29.753772,28.942378,0.429683,0.787980,0.334993,0.512988,0.539361,0.687539
std,0.345200,0.180916,0.269831,34.187300,23.874991,27.639370,0.331878,0.201577,0.272308,0.414714,0.180088,0.236941
min,0.000000,0.125237,0.000000,4.123106,5.000000,4.000000,0.000000,0.120073,0.000000,0.000000,0.085762,0.000000
25%,0.065428,0.568058,0.167989,7.071068,8.660254,9.000000,0.068032,0.737508,0.100089,0.037757,0.443763,0.645833
50%,0.482270,0.686157,0.419162,21.656408,18.027756,14.177447,0.512931,0.839435,0.321744,0.617541,0.578194,0.751374
75%,0.732491,0.755379,0.644857,52.516663,50.408333,43.058102,0.731847,0.916287,0.561488,0.916578,0.645420,0.814925
max,0.888607,0.826484,0.836523,103.639519,78.262383,87.055733,0.894702,0.990487,0.808532,0.992813,0.830151,1.000000


In [84]:
mask = tta_sup_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.413500,0.607751,0.383877,35.020392,31.907179,30.746611,0.439422,0.803947,0.338353,0.473210,0.508500,0.644271
std,0.339330,0.174901,0.273100,34.178273,24.532921,27.707624,0.330135,0.192458,0.280649,0.409501,0.178224,0.250440
min,0.000000,0.147296,0.000000,4.000000,4.358899,4.000000,0.000000,0.169948,0.000000,0.000000,0.082931,0.000000
25%,0.000000,0.542340,0.100818,8.219511,9.105795,8.746287,0.000000,0.728317,0.055258,0.000000,0.410142,0.567270
50%,0.457080,0.663582,0.395173,19.762367,22.610153,20.436868,0.483194,0.876642,0.291534,0.502317,0.542973,0.703249
75%,0.702696,0.731029,0.625706,55.337303,51.151636,50.736702,0.704510,0.937100,0.558362,0.868538,0.619367,0.784629
max,0.888582,0.815566,0.816763,106.898552,78.364532,86.205566,0.902559,0.989233,0.831820,1.000000,0.808733,1.000000


In [85]:
mask = tta_sup_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.261456,0.423896,0.343135,26.765229,32.756437,35.277246,0.230397,0.412010,0.276689,0.697836,0.654263,0.738872
std,0.213247,0.285923,0.288277,21.412703,23.246386,29.447315,0.260135,0.293490,0.254126,0.272820,0.267683,0.291478
min,0.000000,0.000000,0.000000,5.656854,5.099020,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.065754,0.158210,0.021009,14.202254,17.355343,16.805116,0.034740,0.089137,0.010626,0.491920,0.564944,0.669362
50%,0.280158,0.549128,0.293473,22.308077,25.573423,27.459061,0.167284,0.514988,0.196697,0.755881,0.749117,0.796369
75%,0.423865,0.663634,0.607784,28.937377,41.394924,43.378012,0.319517,0.614664,0.532377,0.885387,0.841099,0.955975
max,0.631528,0.866928,0.739580,79.172913,89.553612,107.029198,0.871901,0.852970,0.663668,1.000000,0.884899,1.000000


In [86]:
mask = tta_sup_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.263277,0.425098,0.342643,26.624778,32.757275,35.278646,0.231849,0.412818,0.275900,0.698830,0.654279,0.739155
std,0.214363,0.285114,0.287843,21.417867,23.247346,29.451299,0.260434,0.292633,0.253341,0.273151,0.267725,0.291401
min,0.000000,0.000000,0.000000,5.656854,5.099020,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.066451,0.157738,0.021511,14.171185,17.339387,16.805116,0.035147,0.088765,0.010884,0.493195,0.565084,0.669825
50%,0.282706,0.549081,0.293599,22.226110,25.573423,27.582617,0.167913,0.515024,0.196623,0.762715,0.747952,0.797907
75%,0.423158,0.663934,0.607614,28.953056,41.393881,43.349859,0.319931,0.614756,0.530718,0.883703,0.840789,0.956097
max,0.638296,0.866695,0.737809,79.183334,89.553612,107.029198,0.872491,0.852379,0.659701,1.000000,0.884661,1.000000


In [87]:
mask = tta_sup_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_sup_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.262806,0.423992,0.342823,26.720407,32.752473,35.765037,0.231224,0.412027,0.276412,0.698740,0.654508,0.738908
std,0.213739,0.285902,0.288502,21.402266,23.248514,29.427384,0.259870,0.293448,0.253999,0.273109,0.267695,0.291460
min,0.000000,0.000000,0.000000,5.656854,5.099020,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.065903,0.158728,0.019048,14.196821,17.323399,16.805116,0.034827,0.089224,0.009634,0.492117,0.564830,0.669104
50%,0.281922,0.549111,0.294069,22.037474,25.573423,27.531799,0.167354,0.515172,0.197048,0.755937,0.753547,0.793739
75%,0.423564,0.663950,0.608024,28.953056,41.393881,47.469509,0.318602,0.614575,0.532314,0.883751,0.840802,0.956046
max,0.637114,0.866664,0.737896,79.183334,89.553612,107.029198,0.870720,0.852277,0.660197,1.000000,0.885114,1.000000


## With Consistency

In [88]:
tta_con_lr4_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_lr4_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_lr5_paths =   "/scratch-second/TTA_res_con/val_ent_con_decoder_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_lr5_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_lr6_paths =   "/scratch-second/TTA_res_con/val_ent_con_decoder_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_lr6_augs_paths  =   "/scratch-second/TTA_res_con/val_ent_con_decoder_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [89]:
mask = tta_sup_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_sup_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.413500,0.607751,0.383877,35.020392,31.907179,30.746611,0.439422,0.803947,0.338353,0.473210,0.508500,0.644271
std,0.339330,0.174901,0.273100,34.178273,24.532921,27.707624,0.330135,0.192458,0.280649,0.409501,0.178224,0.250440
min,0.000000,0.147296,0.000000,4.000000,4.358899,4.000000,0.000000,0.169948,0.000000,0.000000,0.082931,0.000000
25%,0.000000,0.542340,0.100818,8.219511,9.105795,8.746287,0.000000,0.728317,0.055258,0.000000,0.410142,0.567270
50%,0.457080,0.663582,0.395173,19.762367,22.610153,20.436868,0.483194,0.876642,0.291534,0.502317,0.542973,0.703249
75%,0.702696,0.731029,0.625706,55.337303,51.151636,50.736702,0.704510,0.937100,0.558362,0.868538,0.619367,0.784629
max,0.888582,0.815566,0.816763,106.898552,78.364532,86.205566,0.902559,0.989233,0.831820,1.000000,0.808733,1.000000


In [90]:
tta_con_lr4 = pd.read_csv(tta_con_lr4_paths)
tta_con_lr4 = clean_df(tta_con_lr4, True, 'brats')
tta_con_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.308899,0.501419,0.395456,38.524546,39.321476,25.295883,0.353946,0.804888,0.328528,0.447450,0.388460,0.704986
std,0.306088,0.176949,0.263640,27.387080,20.875424,26.057943,0.332919,0.225752,0.254231,0.411196,0.167562,0.244506
min,0.000000,0.077428,0.000000,4.000000,7.000000,5.000000,0.000000,0.068328,0.000000,0.000000,0.065714,0.000000
25%,0.001047,0.412741,0.161051,15.211808,20.049938,7.874008,0.001441,0.763010,0.087654,0.000535,0.274222,0.648822
50%,0.232164,0.492214,0.441450,34.727512,39.862263,15.264338,0.262338,0.899486,0.301419,0.446021,0.359304,0.749409
75%,0.587475,0.633534,0.621803,52.412781,55.208694,38.183765,0.660909,0.949674,0.528213,0.918652,0.498466,0.907668
max,0.856765,0.816531,0.852126,103.670151,81.030861,113.021019,0.914928,0.992901,0.828995,1.000000,0.817408,1.000000


In [91]:
tta_con_lr4_augs = pd.read_csv(tta_con_lr4_augs_paths)
tta_con_lr4_augs = clean_df(tta_con_lr4_augs, True, 'brats')
tta_con_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.269535,0.493845,0.494274,26.759774,30.914197,24.754285,0.223693,0.524592,0.448141,0.524454,0.582102,0.746865
std,0.293425,0.220367,0.270842,22.852143,19.736069,25.264243,0.275062,0.293626,0.287841,0.386530,0.217347,0.212305
min,0.000000,0.000000,0.000000,4.472136,9.273619,3.162278,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.001164,0.387211,0.233447,13.300735,15.000000,10.838539,0.000624,0.326390,0.134535,0.105428,0.458559,0.673948
50%,0.100079,0.555311,0.561855,19.899748,26.267851,14.444726,0.054288,0.475795,0.528575,0.639418,0.565527,0.792859
75%,0.505074,0.660644,0.730073,36.045254,40.633148,29.956896,0.404071,0.790571,0.721178,0.899920,0.736825,0.886221
max,0.791639,0.754717,0.847073,116.472313,94.169250,107.916634,0.914116,0.947813,0.865657,0.986826,0.945956,0.992303


In [92]:
tta_con_lr5 = pd.read_csv(tta_con_lr5_paths)
tta_con_lr5 = clean_df(tta_con_lr5, True, 'brats')
tta_con_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.383103,0.602012,0.392677,29.711540,26.290896,27.295305,0.385479,0.734912,0.328252,0.550716,0.530453,0.711054
std,0.312945,0.213933,0.282020,27.742075,21.636909,26.377145,0.319374,0.237281,0.271919,0.400045,0.209991,0.251764
min,0.000000,0.039110,0.000000,4.000000,3.605551,4.000000,0.000000,0.027734,0.000000,0.000000,0.066309,0.000000
25%,0.070316,0.465817,0.097402,9.273619,8.482754,9.689960,0.071945,0.622352,0.051653,0.081570,0.400499,0.639802
50%,0.372897,0.657509,0.441685,19.193363,18.470325,13.269408,0.349677,0.800238,0.296365,0.746969,0.550745,0.764526
75%,0.675053,0.774285,0.643004,41.703603,38.676216,35.665243,0.679038,0.920223,0.550875,0.915454,0.703773,0.866340
max,0.892826,0.908649,0.843031,110.365158,76.563698,92.023094,0.914210,0.990416,0.788140,1.000000,0.880281,1.000000


In [93]:
tta_con_lr5_augs = pd.read_csv(tta_con_lr5_augs_paths)
tta_con_lr5_augs = clean_df(tta_con_lr5_augs, True, 'brats')
tta_con_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.248058,0.422779,0.383119,23.855057,29.685223,26.887103,0.216059,0.372800,0.316931,0.848274,0.704542,0.774998
std,0.233559,0.263556,0.276512,12.123710,17.949972,21.280071,0.258492,0.244276,0.257565,0.189840,0.230675,0.139414
min,0.003652,0.024577,0.004927,6.324555,5.000000,4.582576,0.001830,0.012556,0.002473,0.292723,0.120463,0.503924
25%,0.037501,0.199135,0.076073,13.968652,16.000000,11.806599,0.019317,0.147651,0.040031,0.827530,0.650344,0.661019
50%,0.198235,0.475731,0.439713,25.668859,26.863036,23.816006,0.113070,0.467069,0.363569,0.902702,0.769843,0.796811
75%,0.405267,0.631178,0.602141,30.788708,36.275212,31.226245,0.403520,0.530676,0.496333,0.971750,0.863116,0.888588
max,0.657444,0.859434,0.742624,47.801674,77.932022,88.820045,0.845336,0.816377,0.752373,1.000000,0.913552,0.978942


In [94]:
tta_con_lr6 = pd.read_csv(tta_con_lr6_paths)
tta_con_lr6 = clean_df(tta_con_lr6, True, 'brats')
tta_con_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.364595,0.573547,0.376694,33.393412,30.428351,27.255160,0.414996,0.771165,0.319458,0.451879,0.477034,0.676609
std,0.309564,0.198823,0.283429,31.776733,22.923976,24.578020,0.323329,0.221603,0.277698,0.393405,0.192897,0.291439
min,0.000000,0.127278,0.000000,4.123106,3.741657,4.123106,0.000000,0.117269,0.000000,0.000000,0.082843,0.000000
25%,0.000000,0.443452,0.076323,9.363170,10.254070,9.639409,0.000000,0.656884,0.039863,0.000000,0.339856,0.560504
50%,0.397893,0.597962,0.377217,22.302947,23.959044,16.995697,0.454760,0.851064,0.258439,0.373680,0.467811,0.745992
75%,0.604233,0.723002,0.627341,50.351522,48.538984,36.304099,0.708311,0.932745,0.547511,0.851547,0.604056,0.892580
max,0.889142,0.892874,0.845736,110.763123,78.345390,92.023094,0.974625,0.988613,0.847167,1.000000,0.833558,1.000000


In [95]:
tta_con_lr6_augs = pd.read_csv(tta_con_lr6_augs_paths)
tta_con_lr6_augs = clean_df(tta_con_lr6_augs, True, 'brats')
tta_con_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.240853,0.398534,0.351709,27.652830,31.972622,33.494079,0.211272,0.366662,0.288798,0.716571,0.670674,0.699674
std,0.230306,0.276181,0.301852,20.105091,20.542441,28.859367,0.260954,0.273667,0.272102,0.311424,0.259574,0.307572
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.021027,0.170303,0.030398,13.836804,20.000000,14.910306,0.010798,0.100227,0.015486,0.524034,0.584483,0.673912
50%,0.243947,0.419727,0.341710,24.103799,27.509256,28.239175,0.141443,0.419319,0.244195,0.863688,0.796999,0.782313
75%,0.432221,0.630067,0.626007,33.025236,36.211178,30.953187,0.306026,0.556128,0.525949,0.931778,0.850886,0.881412
max,0.665696,0.865076,0.778833,78.860634,90.232201,106.853874,0.876033,0.842014,0.777482,1.000000,0.889436,1.000000


In [96]:
tta_con_first_layer_lr4_paths =    "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_first_layer_lr4_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_first_layer_lr5_paths =   "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_first_layer_lr5_augs_paths =    "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_first_layer_lr6_paths =   "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_first_layer_lr6_augs_paths  =   "/scratch-second/TTA_res_con/val_ent_con_first_layer_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [97]:
tta_first_layer_lr4 = pd.read_csv(tta_con_first_layer_lr4_paths)
tta_first_layer_lr4 = clean_df(tta_first_layer_lr4, True, 'brats')
tta_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.316604,0.486726,0.338766,40.107355,40.448243,29.419892,0.353317,0.814617,0.278059,0.404294,0.368909,0.705462
std,0.311994,0.195076,0.264769,32.907063,20.867462,27.171857,0.333521,0.239456,0.249013,0.399026,0.190426,0.277845
min,0.000000,0.000000,0.000000,3.605551,5.385165,5.665612,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.373112,0.071673,12.287386,27.648177,8.983407,0.000000,0.784383,0.038629,0.000000,0.257046,0.644891
50%,0.258942,0.489661,0.347696,32.585302,41.707159,17.509565,0.287513,0.903027,0.224040,0.300818,0.329403,0.773348
75%,0.576525,0.608837,0.589219,53.029713,56.876310,41.156469,0.686618,0.954883,0.485453,0.877567,0.454705,0.885606
max,0.886441,0.886727,0.761120,125.231789,81.326500,110.515839,0.937288,0.988304,0.854506,0.985307,0.837047,1.000000


In [98]:
tta_first_layer_lr4_augs = pd.read_csv(tta_con_first_layer_lr4_augs_paths)
tta_first_layer_lr4_augs = clean_df(tta_first_layer_lr4_augs, True, 'brats')
tta_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.219820,0.483792,0.417432,22.473812,32.769952,29.219918,0.210216,0.476616,0.339774,0.525458,0.591295,0.752884
std,0.252483,0.222951,0.258386,11.173940,18.072449,27.022041,0.284278,0.266070,0.249594,0.346296,0.207073,0.235473
min,0.000000,0.063938,0.000000,6.164414,8.774964,6.480741,0.000000,0.033637,0.000000,0.000000,0.063832,0.000000
25%,0.005073,0.385530,0.204083,14.182166,19.714670,11.500000,0.003000,0.292684,0.115735,0.305039,0.431163,0.687490
50%,0.126553,0.491156,0.461232,24.175796,30.532225,19.949340,0.077641,0.514740,0.384783,0.608204,0.644886,0.810270
75%,0.399450,0.663955,0.651131,27.774435,43.241027,34.408051,0.328863,0.697699,0.515896,0.792835,0.723086,0.915100
max,0.729095,0.788274,0.777449,50.950958,76.193176,107.582527,0.826871,0.977458,0.767143,1.000000,0.902595,0.998125


In [99]:
tta_first_layer_lr5 = pd.read_csv(tta_con_first_layer_lr5_paths)
tta_first_layer_lr5 = clean_df(tta_first_layer_lr5, True, 'brats')
tta_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.381125,0.594208,0.387372,29.765170,28.666055,27.028144,0.381714,0.738274,0.322334,0.550199,0.518468,0.730815
std,0.312355,0.204133,0.283520,27.578315,22.643244,26.123578,0.317930,0.238819,0.271693,0.404022,0.197374,0.253655
min,0.000000,0.046104,0.000000,3.741657,3.316625,3.741657,0.000000,0.033365,0.000000,0.000000,0.074575,0.000000
25%,0.066224,0.467589,0.103899,8.366600,9.000000,9.899495,0.034483,0.624701,0.055271,0.049680,0.399956,0.670856
50%,0.402818,0.611180,0.435392,20.124611,21.095022,13.453624,0.321526,0.809157,0.286326,0.780878,0.506664,0.778017
75%,0.667847,0.760726,0.650691,42.766811,46.094460,32.634338,0.643282,0.929134,0.545404,0.919396,0.686168,0.908230
max,0.896078,0.904776,0.845880,107.122353,77.935867,92.023094,0.908409,0.993251,0.786233,1.000000,0.864438,1.000000


In [100]:
tta_first_layer_lr5_augs = pd.read_csv(tta_con_first_layer_lr5_augs_paths)
tta_first_layer_lr5_augs = clean_df(tta_first_layer_lr5_augs, True, 'brats')
tta_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.229752,0.404913,0.367297,27.338323,34.149465,30.227464,0.196809,0.359437,0.308467,0.820867,0.663901,0.719183
std,0.237619,0.275659,0.287291,17.877520,26.497133,24.546393,0.248754,0.255029,0.272768,0.268704,0.274474,0.232857
min,0.000000,0.000000,0.000000,5.757513,5.000000,4.358899,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.017586,0.158622,0.070616,18.020815,17.516648,14.109577,0.008877,0.114958,0.037153,0.840763,0.588808,0.623170
50%,0.154088,0.416764,0.400887,26.000000,26.772186,25.658323,0.083475,0.447791,0.305800,0.912162,0.708329,0.777543
75%,0.369390,0.626319,0.591772,32.856027,40.991594,32.137970,0.330464,0.545637,0.480658,0.990466,0.858492,0.859456
max,0.660089,0.864553,0.780953,82.419655,115.990730,88.831871,0.776860,0.830634,0.768360,1.000000,0.917456,0.978894


In [101]:
tta_first_layer_lr6 = pd.read_csv(tta_con_first_layer_lr6_paths)
tta_first_layer_lr6 = clean_df(tta_first_layer_lr6, True, 'brats')
tta_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.366522,0.573862,0.376615,33.322725,30.393767,27.282331,0.416420,0.771747,0.318805,0.452362,0.477083,0.678110
std,0.310543,0.198287,0.283719,31.794929,22.852777,24.636686,0.323524,0.222097,0.277185,0.393593,0.192307,0.290851
min,0.000000,0.126959,0.000000,4.123106,3.741657,4.123106,0.000000,0.116498,0.000000,0.000000,0.082747,0.000000
25%,0.000094,0.443744,0.076675,9.294291,10.218068,9.650490,0.002006,0.657230,0.040056,0.000048,0.340089,0.564563
50%,0.401507,0.600038,0.381035,22.302947,23.959386,16.995697,0.459736,0.850583,0.261221,0.375661,0.467233,0.750608
75%,0.607444,0.722702,0.628299,50.405675,48.543793,36.582091,0.710101,0.937934,0.546336,0.857557,0.603878,0.892291
max,0.889524,0.892212,0.845267,110.763123,78.313469,92.023094,0.974625,0.989018,0.846894,1.000000,0.832090,1.000000


In [102]:
tta_first_layer_lr6_augs = pd.read_csv(tta_con_first_layer_lr6_augs_paths)
tta_first_layer_lr6_augs = clean_df(tta_first_layer_lr6_augs, True, 'brats')
tta_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.240591,0.398824,0.352266,27.611795,31.978993,33.454027,0.211330,0.366885,0.289121,0.718998,0.669881,0.700183
std,0.231215,0.276317,0.301500,20.121830,20.548569,28.865917,0.262098,0.273686,0.271661,0.310150,0.258801,0.307568
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.021876,0.169870,0.031125,13.627626,20.000000,14.884147,0.011232,0.100312,0.015864,0.538078,0.582924,0.672535
50%,0.240236,0.420238,0.343079,24.156760,27.509256,28.265503,0.138450,0.419084,0.245596,0.864070,0.789841,0.785746
75%,0.426604,0.630341,0.628457,32.827381,36.211178,30.940794,0.305712,0.556337,0.519810,0.931588,0.851083,0.875520
max,0.669457,0.865240,0.778689,78.894234,90.277351,106.853874,0.879575,0.842708,0.775535,1.000000,0.889011,1.000000


In [103]:
tta_con_norm_lr4_paths =    "/scratch-second/TTA_results/val_ent_con_norm_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_norm_lr4_augs_paths =    "/scratch-second/TTA_results/val_ent_con_norm_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_norm_lr5_paths =   "/scratch-second/TTA_results/val_ent_con_norm_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_norm_lr5_augs_paths =    "/scratch-second/TTA_results/val_ent_con_norm_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_norm_lr6_paths =   "/scratch-second/TTA_results/val_ent_con_norm_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_con_norm_lr6_augs_paths  =   "/scratch-second/TTA_results/val_ent_con_norm_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"


In [104]:
tta_con_norm_lr4 = pd.read_csv(tta_con_norm_lr4_paths)
tta_con_norm_lr4 = clean_df(tta_con_norm_lr4, True, 'brats')
tta_con_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.312720,0.489651,0.339588,34.148540,40.433574,29.863229,0.352311,0.809451,0.279986,0.451741,0.371840,0.687150
std,0.299942,0.194813,0.265407,26.146607,21.144977,27.563099,0.336347,0.242607,0.251751,0.399951,0.190042,0.281473
min,0.000000,0.000000,0.000000,4.000000,7.211102,5.099020,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.022085,0.435579,0.074212,12.000000,23.640080,8.942121,0.011860,0.757448,0.044202,0.031393,0.280989,0.619136
50%,0.247501,0.489338,0.331051,25.989085,42.672764,17.436913,0.300734,0.900605,0.210622,0.400137,0.345697,0.754120
75%,0.551772,0.601312,0.569061,51.367947,56.570492,40.022488,0.712752,0.964472,0.520071,0.900966,0.461292,0.875602
max,0.868197,0.871517,0.795954,98.488571,82.758682,111.184532,0.942149,0.991163,0.865314,0.997555,0.832044,1.000000


In [105]:
tta_con_norm_lr4_augs = pd.read_csv(tta_con_norm_lr4_augs_paths)
tta_con_norm_lr4_augs = clean_df(tta_con_norm_lr4_augs, True, 'brats')
tta_con_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000,23.000000
mean,0.182622,0.478578,0.405835,30.116529,34.438080,30.026343,0.170368,0.456959,0.325519,0.401259,0.578613,0.749141
std,0.238321,0.247500,0.269889,25.450866,24.764911,28.026672,0.248400,0.281654,0.252599,0.355492,0.263297,0.276580
min,0.000000,0.000000,0.000000,6.164414,9.433981,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.345108,0.147471,15.905477,19.500000,11.468845,0.000000,0.240256,0.079989,0.000000,0.495551,0.697628
50%,0.007151,0.562979,0.412790,23.259407,26.000000,16.890785,0.003588,0.473472,0.295088,0.413155,0.641590,0.806510
75%,0.337565,0.613848,0.639049,36.735126,39.323393,38.702564,0.220516,0.666457,0.541561,0.667095,0.760994,0.935588
max,0.703282,0.820043,0.793280,119.968330,100.347137,107.413925,0.780340,0.970843,0.811185,1.000000,0.905710,0.995896


In [106]:
tta_con_norm_lr5 = pd.read_csv(tta_con_norm_lr5_paths)
tta_con_norm_lr5 = clean_df(tta_con_norm_lr5, True, 'brats')
tta_con_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.372730,0.588883,0.385885,30.615562,29.880306,27.121887,0.376107,0.732333,0.322305,0.536677,0.512776,0.726202
std,0.314857,0.207351,0.285244,28.350992,22.991117,26.241640,0.321971,0.242841,0.273474,0.404121,0.199485,0.257885
min,0.000000,0.044707,0.000000,3.922421,3.605551,3.741657,0.000000,0.031718,0.000000,0.000000,0.075710,0.000000
25%,0.041667,0.467468,0.079916,10.000000,9.433981,8.774964,0.021552,0.620435,0.041952,0.042759,0.380558,0.647922
50%,0.350842,0.618051,0.434808,19.919804,23.108440,13.601471,0.338833,0.799763,0.286196,0.625000,0.506018,0.782549
75%,0.667072,0.739993,0.641069,43.104523,49.406479,33.139103,0.661333,0.922588,0.542996,0.920245,0.645028,0.904469
max,0.895262,0.906437,0.845473,110.546227,78.162003,92.023094,0.910062,0.991774,0.789502,1.000000,0.870004,1.000000


In [107]:
tta_con_norm_lr5_augs = pd.read_csv(tta_con_norm_lr5_augs_paths)
tta_con_norm_lr5_augs = clean_df(tta_con_norm_lr5_augs, True, 'brats')
tta_con_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.241752,0.422781,0.377490,23.864796,30.441735,28.103741,0.206435,0.372951,0.305184,0.866042,0.702070,0.765676
std,0.233659,0.263305,0.272293,12.160173,18.063469,21.220531,0.247986,0.245036,0.242841,0.188051,0.228590,0.157323
min,0.003457,0.024834,0.002138,5.830952,5.000000,4.358899,0.001732,0.012700,0.001072,0.308166,0.112041,0.450867
25%,0.038044,0.191484,0.078612,13.999848,16.274972,16.730230,0.019531,0.148189,0.041399,0.857039,0.633434,0.686894
50%,0.183125,0.472696,0.449864,25.396851,29.546217,24.082720,0.101127,0.431813,0.355526,0.913782,0.769676,0.795464
75%,0.405283,0.632843,0.576201,30.630779,38.384685,31.131870,0.400305,0.538987,0.489761,0.989919,0.860010,0.887993
max,0.648227,0.867832,0.744673,48.105606,77.932022,88.831871,0.768595,0.836208,0.680861,1.000000,0.907882,0.977374


In [108]:
tta_con_norm_lr6_augs = pd.read_csv(tta_con_norm_lr6_augs_paths)
tta_con_norm_lr6_augs = clean_df(tta_con_norm_lr6_augs, True, 'brats')
tta_con_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000,22.000000
mean,0.240461,0.398794,0.352015,27.634824,31.970459,33.491075,0.210988,0.366982,0.289118,0.717798,0.669991,0.699721
std,0.230265,0.276238,0.301796,20.120009,20.537854,28.860503,0.261268,0.273791,0.272076,0.310863,0.259092,0.307470
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.123106,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.021197,0.170124,0.030642,13.710605,20.000000,14.884147,0.010886,0.100330,0.015613,0.527378,0.581906,0.671874
50%,0.241364,0.420858,0.343031,24.114202,27.509256,28.265503,0.139185,0.419774,0.245548,0.863194,0.792337,0.783370
75%,0.428699,0.629917,0.626535,33.013192,36.211178,30.940794,0.304644,0.555641,0.522251,0.932007,0.850849,0.877430
max,0.665049,0.865208,0.778717,78.873314,90.259903,106.853874,0.876623,0.842643,0.777392,1.000000,0.889015,1.000000


In [109]:
tta_con_norm_lr6 = pd.read_csv(tta_con_norm_lr6_paths)
tta_con_norm_lr6 = clean_df(tta_con_norm_lr6, True, 'brats')
tta_con_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.364863,0.573895,0.376650,33.366843,30.406958,27.286548,0.415346,0.771842,0.319168,0.451709,0.477086,0.677446
std,0.309887,0.198216,0.283633,31.768182,22.868702,24.628863,0.323599,0.221883,0.277505,0.393454,0.192303,0.291262
min,0.000000,0.126805,0.000000,4.123106,3.741657,4.123106,0.000000,0.116315,0.000000,0.000000,0.082861,0.000000
25%,0.000000,0.443690,0.076079,9.342881,10.219826,9.650490,0.000000,0.657093,0.039729,0.000000,0.339471,0.562142
50%,0.399912,0.599014,0.379440,22.302947,24.039606,16.995697,0.457227,0.851065,0.260115,0.373611,0.467269,0.748907
75%,0.605797,0.722704,0.627411,50.391513,48.535013,36.560505,0.709982,0.937718,0.547079,0.853552,0.603973,0.892409
max,0.889284,0.892503,0.845607,110.763123,78.319855,92.023094,0.974625,0.988780,0.847167,1.000000,0.832697,1.000000


In [110]:
mask = tta_con_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_con_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.398813,0.529935,0.418041,34.887367,40.214312,26.207798,0.451349,0.844692,0.363435,0.524569,0.409572,0.674249
std,0.327574,0.167569,0.264667,26.787275,21.951498,27.926907,0.341906,0.173149,0.270350,0.418283,0.176611,0.238871
min,0.000000,0.119377,0.000000,4.000000,7.549834,5.000000,0.000000,0.187761,0.000000,0.000000,0.065714,0.000000
25%,0.019171,0.434392,0.247709,9.899495,19.723083,7.280110,0.009678,0.804496,0.161547,0.046972,0.286842,0.562364
50%,0.403662,0.494986,0.396193,29.017237,41.424629,13.453624,0.495812,0.915466,0.363625,0.638380,0.359304,0.725324
75%,0.752336,0.657340,0.651400,52.354561,59.539902,39.440445,0.764076,0.955399,0.591151,0.968087,0.502047,0.787957
max,0.856765,0.816531,0.852126,103.670151,78.192070,113.021019,0.914928,0.992901,0.828995,1.000000,0.817408,0.990136


In [111]:
mask = tta_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.397599,0.523080,0.366936,36.548923,41.931195,29.065111,0.439337,0.864909,0.313230,0.474702,0.399050,0.699692
std,0.326957,0.163777,0.262609,33.557418,21.797707,28.004372,0.346278,0.164721,0.262277,0.402562,0.173972,0.245692
min,0.000000,0.104685,0.000000,3.605551,7.071068,5.665612,0.000000,0.138594,0.000000,0.000000,0.055993,0.000000
25%,0.074712,0.451067,0.115598,9.110434,25.670996,8.306623,0.087627,0.829461,0.062055,0.039596,0.298878,0.631543
50%,0.357458,0.517009,0.363113,26.314442,47.360321,16.552946,0.437206,0.941717,0.242417,0.501847,0.373280,0.718656
75%,0.670806,0.609779,0.601060,50.388493,58.215118,41.107780,0.783738,0.965534,0.491529,0.915924,0.466897,0.864512
max,0.886441,0.830982,0.761120,125.231789,77.414467,110.515839,0.937288,0.978789,0.854506,0.985307,0.825973,1.000000


In [112]:
mask = tta_con_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_con_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.397481,0.520365,0.375258,31.384302,42.601111,28.957343,0.431773,0.853099,0.323222,0.535063,0.398104,0.686443
std,0.312840,0.163592,0.261689,24.912265,21.624404,28.021290,0.330998,0.176047,0.264170,0.395498,0.177444,0.248870
min,0.000000,0.096500,0.000000,4.000000,7.211102,5.099020,0.000000,0.111492,0.000000,0.000000,0.067775,0.000000
25%,0.080457,0.448132,0.106058,9.848858,24.535688,7.874008,0.098993,0.773346,0.059660,0.137046,0.300980,0.646028
50%,0.366265,0.513353,0.338809,25.475479,47.265747,16.278820,0.446956,0.934495,0.251513,0.615640,0.366349,0.722377
75%,0.672520,0.628028,0.613341,51.138042,58.532043,40.012497,0.769048,0.966340,0.545885,0.943178,0.490141,0.839641
max,0.868197,0.808321,0.795954,98.488571,77.781746,111.184532,0.891938,0.986279,0.865314,0.997555,0.832044,1.000000


In [113]:
mask = tta_con_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.243819,0.415962,0.325751,28.207390,32.383490,36.512041,0.217211,0.390741,0.258779,0.728526,0.665561,0.681915
std,0.213103,0.280815,0.284950,21.203011,22.094344,29.738157,0.262937,0.279424,0.246833,0.278660,0.270639,0.328069
min,0.000000,0.000000,0.000000,5.744563,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.056700,0.171829,0.028191,14.985031,17.405694,16.791562,0.029391,0.102820,0.014331,0.568787,0.591383,0.628840
50%,0.259745,0.505711,0.292393,24.020824,27.018513,29.698484,0.152924,0.458248,0.196076,0.845791,0.796137,0.779167
75%,0.404430,0.653658,0.608051,32.403782,41.418274,43.678036,0.278407,0.570169,0.496573,0.930379,0.849744,0.911112
max,0.630943,0.865076,0.733557,78.860634,90.232201,106.853874,0.876033,0.842014,0.651767,1.000000,0.889436,1.000000


In [114]:
mask = tta_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.243286,0.416200,0.326395,28.193116,32.384232,36.465666,0.216983,0.390922,0.259296,0.731449,0.664501,0.682161
std,0.213875,0.280999,0.284592,21.221009,22.103127,29.750534,0.263931,0.279472,0.246615,0.276949,0.269904,0.327959
min,0.000000,0.000000,0.000000,5.744563,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.055258,0.171583,0.031537,14.845579,17.421490,16.791562,0.028590,0.102902,0.016054,0.580114,0.590839,0.627044
50%,0.253732,0.505923,0.294017,24.147429,27.018513,29.017237,0.147858,0.460027,0.197669,0.847335,0.782237,0.783900
75%,0.399078,0.653868,0.609280,32.323345,41.418274,43.678036,0.275926,0.570900,0.497282,0.929667,0.847892,0.905785
max,0.634204,0.865240,0.734033,78.894234,90.277351,106.853874,0.879575,0.842708,0.653751,1.000000,0.889011,1.000000


In [115]:
mask = tta_con_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.243378,0.416199,0.326103,28.190581,32.380985,36.508563,0.216911,0.391040,0.259157,0.729952,0.664772,0.681825
std,0.213153,0.280883,0.284907,21.221228,22.089418,29.739816,0.263367,0.279567,0.246859,0.277867,0.270115,0.327902
min,0.000000,0.000000,0.000000,5.744563,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.056264,0.171754,0.028879,14.891246,17.421490,16.791562,0.029127,0.102927,0.014688,0.571951,0.589861,0.627253
50%,0.254360,0.506394,0.293096,24.041630,27.018513,29.681644,0.148278,0.458987,0.196648,0.845934,0.787293,0.777254
75%,0.401316,0.653622,0.608389,32.379694,41.418274,43.678036,0.275782,0.570095,0.497185,0.930335,0.848868,0.907614
max,0.633744,0.865208,0.734129,78.873314,90.259903,106.853874,0.876623,0.842643,0.653680,1.000000,0.889015,1.000000


In [116]:
mask = tta_con_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.358413,0.496656,0.486719,24.779584,33.162920,28.074245,0.303289,0.546563,0.440084,0.659606,0.535014,0.711350
std,0.291969,0.234264,0.259088,26.121770,21.983708,28.352569,0.289158,0.290660,0.278677,0.331404,0.228041,0.224185
min,0.000000,0.000000,0.000000,4.472136,9.273619,6.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.061135,0.431487,0.292871,10.325486,15.000000,12.243737,0.034889,0.379036,0.182552,0.479421,0.448863,0.620323
50%,0.360021,0.580395,0.562546,17.017938,33.254934,14.740667,0.277611,0.601624,0.539008,0.843958,0.504496,0.752577
75%,0.586668,0.663627,0.697060,22.373476,44.201542,30.678766,0.466263,0.788622,0.704372,0.903362,0.692858,0.833876
max,0.791639,0.732394,0.767446,116.472313,94.169250,107.916634,0.914116,0.947813,0.787238,0.949651,0.898813,0.992303


In [117]:
mask = tta_first_layer_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_first_layer_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.289844,0.457598,0.364880,22.215894,37.356363,35.423562,0.278598,0.476633,0.284913,0.605174,0.557539,0.738464
std,0.255859,0.237093,0.258538,12.331177,18.570199,28.715620,0.299377,0.298561,0.237775,0.303879,0.206954,0.239645
min,0.000000,0.063938,0.000000,6.164414,8.774964,6.480741,0.000000,0.033637,0.000000,0.000000,0.063832,0.000000
25%,0.012910,0.300823,0.123053,13.123903,25.688779,16.127234,0.006595,0.209044,0.067295,0.363734,0.424322,0.696579
50%,0.247669,0.483467,0.292646,24.000000,38.091995,29.899834,0.176150,0.561006,0.184858,0.662073,0.644807,0.792839
75%,0.453208,0.633173,0.615723,26.836546,48.122463,36.387472,0.439739,0.706828,0.495035,0.833390,0.669144,0.880130
max,0.729095,0.788274,0.702976,50.950958,76.193176,107.582527,0.826871,0.977458,0.716884,1.000000,0.831716,0.998125


In [118]:
mask = tta_con_norm_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_norm_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.266257,0.428296,0.338696,32.211298,42.188939,39.860600,0.253576,0.430788,0.261509,0.454446,0.511195,0.701918
std,0.255599,0.273467,0.264011,30.838154,27.692891,30.462607,0.273663,0.322573,0.226828,0.357346,0.290746,0.306837
min,0.000000,0.000000,0.000000,6.164414,9.433981,8.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.003575,0.230608,0.098685,14.389867,26.416408,16.695775,0.001794,0.140950,0.052391,0.142828,0.410179,0.682479
50%,0.245702,0.450516,0.412790,23.259407,38.340580,34.581059,0.197383,0.442142,0.290359,0.413155,0.579229,0.803101
75%,0.429760,0.605353,0.571733,36.735126,48.774986,50.251963,0.509931,0.657317,0.446800,0.720672,0.739098,0.872479
max,0.703282,0.820043,0.687037,119.968330,100.347137,107.413925,0.780340,0.970843,0.590015,1.000000,0.832659,0.995896


In [119]:
mask = tta_con_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000
mean,0.262434,0.445855,0.404015,23.408392,29.392008,26.819539,0.228661,0.393817,0.334740,0.839349,0.695675,0.772634
std,0.232393,0.252228,0.269976,12.343226,18.457921,21.933005,0.260686,0.234420,0.253812,0.191751,0.234592,0.143333
min,0.005705,0.024577,0.004927,6.324555,5.000000,4.582576,0.002861,0.012556,0.002473,0.292723,0.120463,0.503924
25%,0.074822,0.225529,0.118128,12.000000,15.000000,11.357817,0.039191,0.203348,0.063533,0.823402,0.648368,0.647482
50%,0.226744,0.524644,0.458420,25.337719,26.248810,22.000000,0.132879,0.480903,0.365618,0.895821,0.726369,0.778440
75%,0.434862,0.640730,0.612172,28.809721,36.810326,31.780497,0.474222,0.535983,0.515320,0.942349,0.865284,0.900094
max,0.657444,0.859434,0.742624,47.801674,77.932022,88.820045,0.845336,0.816377,0.752373,1.000000,0.913552,0.978942


In [120]:
mask = tta_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.242309,0.425705,0.386054,27.109987,34.110963,30.350220,0.207640,0.378539,0.324765,0.810915,0.653292,0.714181
std,0.237932,0.267881,0.283396,18.367286,27.264778,25.252028,0.251314,0.248042,0.270990,0.272868,0.278394,0.238555
min,0.000000,0.000000,0.000000,5.757513,5.000000,4.358899,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.030219,0.197058,0.095839,15.010407,16.274972,12.842707,0.015462,0.163805,0.050973,0.840213,0.572613,0.620553
50%,0.159113,0.478360,0.454348,25.698425,26.662011,23.930391,0.086683,0.472952,0.335322,0.905581,0.707579,0.769316
75%,0.403409,0.635371,0.608428,32.755559,43.090045,32.146919,0.402425,0.553502,0.505488,0.975541,0.860126,0.866305
max,0.660089,0.864553,0.780953,82.419655,115.990730,88.831871,0.776860,0.830634,0.768360,1.000000,0.917456,0.978894


In [121]:
mask = tta_con_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_con_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000
mean,0.255770,0.445857,0.398034,23.418703,30.178188,28.107744,0.218476,0.393976,0.322292,0.858163,0.693118,0.762918
std,0.232919,0.251950,0.265905,12.381673,18.583692,21.873616,0.250135,0.235247,0.238874,0.190750,0.232350,0.161716
min,0.003727,0.024834,0.002138,5.830952,5.000000,4.358899,0.001867,0.012700,0.001072,0.308166,0.112041,0.450867
25%,0.065488,0.226970,0.118687,12.041595,15.033297,16.640306,0.034086,0.201988,0.063824,0.846311,0.621856,0.681892
50%,0.189232,0.508588,0.483874,25.396851,27.092434,22.494444,0.105150,0.453088,0.389384,0.911563,0.724526,0.778361
75%,0.439939,0.636147,0.580022,28.178005,38.910152,31.654383,0.471624,0.546325,0.504633,0.987075,0.860494,0.899711
max,0.648227,0.867832,0.744673,48.105606,77.932022,88.831871,0.768595,0.836208,0.680861,1.000000,0.907882,0.977374


In [122]:
mask = tta_con_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_con_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.444127,0.622491,0.398568,28.693732,28.909614,28.499214,0.445208,0.769547,0.338394,0.558684,0.548441,0.682742
std,0.334204,0.204558,0.272228,29.029903,22.675813,27.988837,0.327379,0.225750,0.270523,0.419422,0.205056,0.250831
min,0.000000,0.039110,0.000000,4.000000,3.605551,4.000000,0.000000,0.027734,0.000000,0.000000,0.066309,0.000000
25%,0.075203,0.562435,0.176142,9.273619,8.124039,8.774964,0.123005,0.749677,0.098083,0.040158,0.422642,0.559190
50%,0.473415,0.690033,0.432147,17.521416,22.912878,12.688578,0.403565,0.835057,0.286715,0.831356,0.583716,0.743475
75%,0.742045,0.768934,0.640430,41.130882,46.441895,45.978256,0.782226,0.918517,0.545637,0.935237,0.702954,0.841498
max,0.892826,0.894134,0.843031,110.365158,73.558144,87.134956,0.914210,0.990416,0.788140,1.000000,0.857847,1.000000


In [123]:
mask = tta_con_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_con_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.444242,0.613570,0.400053,29.340791,32.188492,28.404826,0.447143,0.775677,0.339737,0.558585,0.533470,0.683829
std,0.332550,0.197314,0.272725,30.225267,24.261099,27.995189,0.327761,0.225808,0.271160,0.420541,0.196338,0.250408
min,0.000000,0.044707,0.000000,3.922421,3.605551,3.741657,0.000000,0.031718,0.000000,0.000000,0.075710,0.000000
25%,0.074520,0.558161,0.170115,9.219544,8.944272,8.062258,0.124778,0.760555,0.094362,0.042759,0.423362,0.567735
50%,0.456609,0.681671,0.434808,17.492855,24.718414,12.688578,0.421916,0.849752,0.286196,0.840635,0.575915,0.751689
75%,0.746425,0.762080,0.641069,42.305416,53.525696,45.011108,0.774052,0.922588,0.542996,0.941179,0.688357,0.844636
max,0.895262,0.819240,0.845473,110.546227,78.162003,87.143562,0.910062,0.991774,0.789502,0.999531,0.834882,1.000000


In [124]:
mask = tta_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.452905,0.617859,0.401177,27.998707,31.659531,28.593684,0.454722,0.784760,0.339294,0.565228,0.535121,0.692784
std,0.327684,0.190473,0.271561,28.977974,23.748984,27.815066,0.321919,0.215086,0.269682,0.416004,0.193662,0.245139
min,0.000000,0.046104,0.000000,3.741657,3.316625,3.741657,0.000000,0.033365,0.000000,0.000000,0.074575,0.000000
25%,0.074627,0.565119,0.142175,7.892851,9.000000,9.813169,0.253185,0.766877,0.077127,0.049680,0.422625,0.580441
50%,0.462370,0.681297,0.435392,16.907091,24.718414,13.000000,0.428382,0.858074,0.286326,0.844147,0.567868,0.744580
75%,0.756354,0.760726,0.636572,40.934090,50.621140,45.000000,0.776672,0.936635,0.545404,0.936912,0.686168,0.839955
max,0.896078,0.816973,0.845880,107.122353,77.935867,87.143562,0.908409,0.993251,0.786233,0.998091,0.831245,1.000000


In [125]:
mask = tta_con_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_con_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.428820,0.610794,0.396765,34.442741,32.639259,28.617782,0.461648,0.812398,0.348568,0.490242,0.510593,0.648097
std,0.333898,0.179313,0.266971,35.154850,24.473158,26.294084,0.324636,0.190994,0.277318,0.406924,0.183816,0.259799
min,0.000000,0.147194,0.000000,4.123106,4.123106,4.123106,0.000000,0.151465,0.000000,0.000000,0.082843,0.000000
25%,0.037341,0.555327,0.147574,7.754014,8.910935,8.954877,0.124170,0.772243,0.082888,0.019784,0.400849,0.560169
50%,0.416015,0.669189,0.421349,18.980249,25.099800,14.764823,0.499412,0.884345,0.337913,0.644873,0.548092,0.716778
75%,0.735921,0.733195,0.629729,56.382595,53.508600,46.185163,0.739402,0.934224,0.552934,0.868087,0.630149,0.787533
max,0.889142,0.814336,0.820962,110.763123,78.345390,86.236305,0.901783,0.988613,0.847167,0.978680,0.817284,1.000000


In [126]:
mask = tta_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.431826,0.610965,0.396736,34.376799,32.656811,28.649689,0.464356,0.813651,0.347523,0.490105,0.510439,0.650426
std,0.334807,0.179116,0.267217,35.195236,24.514185,26.379383,0.324638,0.191286,0.276575,0.406904,0.183669,0.259736
min,0.000000,0.147069,0.000000,4.123106,4.123106,4.123106,0.000000,0.150948,0.000000,0.000000,0.082747,0.000000
25%,0.037236,0.558111,0.147236,7.658639,8.883739,8.986756,0.146151,0.772839,0.082684,0.019728,0.401108,0.563204
50%,0.449404,0.669072,0.421445,18.814888,25.079872,14.899665,0.499608,0.884770,0.337676,0.642857,0.547675,0.719473
75%,0.736874,0.733781,0.629907,56.432014,53.539885,46.211084,0.741906,0.937977,0.548252,0.867450,0.630960,0.790143
max,0.889524,0.814004,0.823197,110.763123,78.313469,87.057449,0.901325,0.989018,0.846894,0.980759,0.817138,1.000000


In [127]:
mask = tta_con_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_con_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.429452,0.611003,0.396695,34.418887,32.681658,28.648436,0.462539,0.813401,0.348017,0.489886,0.510517,0.649202
std,0.334364,0.178970,0.267025,35.170300,24.531527,26.371537,0.324994,0.191151,0.276962,0.406905,0.183556,0.259803
min,0.000000,0.147255,0.000000,4.123106,4.123106,4.123106,0.000000,0.152289,0.000000,0.000000,0.082861,0.000000
25%,0.037280,0.557944,0.148454,7.713436,8.883739,8.983550,0.127744,0.773362,0.083428,0.019750,0.401555,0.561674
50%,0.412913,0.668724,0.422052,18.867962,25.099800,14.764823,0.498726,0.884709,0.338365,0.644298,0.547334,0.718878
75%,0.736737,0.733531,0.629764,56.431276,53.723530,46.228947,0.740291,0.937808,0.550589,0.867997,0.630645,0.788784
max,0.889284,0.814181,0.821938,110.763123,78.319855,87.055733,0.901572,0.988780,0.847167,0.979153,0.817232,1.000000


## ENT AND SUPERVISION

In [128]:
tta_combined_decoder_lr4_paths =    "/scratch-second/TTA_results/val_combined_decoder_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_decoder_lr4_augs_paths =    "/scratch-second/TTA_results/val_combined_decoder_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_decoder_lr5_paths =   "/scratch-second/TTA_results/val_combined_decoder_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_decoder_lr5_augs_paths =    "/scratch-second/TTA_results/val_combined_decoder_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_decoder_lr6_paths =   "/scratch-second/TTA_results/val_combined_decoder_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_decoder_lr6_augs_paths  =   "/scratch-second/TTA_results/val_combined_decoder_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [129]:
tta_combined_decoder_lr6 = pd.read_csv(tta_combined_decoder_lr6_paths)
tta_combined_decoder_lr6 = clean_df(tta_combined_decoder_lr6, True, 'brats')
tta_combined_decoder_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.376948,0.573890,0.378849,34.410009,29.386360,27.633985,0.429586,0.773836,0.321121,0.452615,0.476414,0.680370
std,0.304958,0.199990,0.284868,30.488655,23.200180,24.769874,0.314470,0.220984,0.278248,0.385402,0.193705,0.282374
min,0.000000,0.131657,0.000000,4.472136,3.741657,4.000000,0.000000,0.121951,0.000000,0.000000,0.084113,0.000000
25%,0.041361,0.435499,0.067402,9.374250,10.459789,9.219060,0.088928,0.662952,0.035175,0.021287,0.345361,0.564395
50%,0.401896,0.598254,0.395869,23.687922,19.759610,17.711432,0.475410,0.847073,0.276979,0.360574,0.465505,0.736454
75%,0.620488,0.739619,0.620918,54.845939,48.378694,40.834575,0.706193,0.934816,0.543461,0.825463,0.619247,0.882489
max,0.890360,0.891465,0.848307,106.714561,78.497757,92.023094,0.967705,0.989280,0.857610,0.996016,0.831770,1.000000


In [130]:
tta_combined_decoder_lr6_augs = pd.read_csv(tta_combined_decoder_lr6_augs_paths)
tta_combined_decoder_lr6_augs = clean_df(tta_combined_decoder_lr6_augs, True, 'brats')
tta_combined_decoder_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.272602,0.420924,0.352583,26.445701,31.797036,32.476636,0.230082,0.387774,0.290313,0.688160,0.666558,0.723708
std,0.227898,0.271017,0.302043,20.383650,19.944990,26.399797,0.250765,0.268809,0.274552,0.329743,0.256081,0.280542
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.044339,0.173040,0.027970,11.312284,20.527024,15.722172,0.023274,0.132702,0.014242,0.519791,0.547031,0.674258
50%,0.291032,0.442048,0.353686,22.963835,27.000000,27.766513,0.181129,0.432553,0.244855,0.833206,0.801988,0.785911
75%,0.488613,0.657235,0.606621,28.959062,37.243983,33.790886,0.350834,0.567198,0.516531,0.925972,0.841841,0.910465
max,0.661615,0.868435,0.798786,78.813705,90.887566,107.029198,0.894923,0.857119,0.788627,1.000000,0.924886,1.000000


In [131]:
tta_combined_decoder_lr5 = pd.read_csv(tta_combined_decoder_lr5_paths)
tta_combined_decoder_lr5 = clean_df(tta_combined_decoder_lr5, True, 'brats')
tta_combined_decoder_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.381858,0.593439,0.373499,31.898222,26.374084,27.932796,0.401504,0.755636,0.309190,0.510107,0.506345,0.700522
std,0.307400,0.199977,0.283880,30.560821,21.636977,25.968357,0.314066,0.227140,0.270263,0.396075,0.189101,0.281885
min,0.000000,0.069303,0.000000,3.308905,4.000000,4.000000,0.000000,0.059707,0.000000,0.000000,0.082574,0.000000
25%,0.063440,0.473770,0.075795,7.810250,8.831760,9.949874,0.086320,0.694749,0.039599,0.033876,0.375955,0.617425
50%,0.370623,0.655209,0.404003,20.904545,17.962454,17.233688,0.371625,0.833782,0.286168,0.560603,0.534286,0.791441
75%,0.637384,0.748998,0.637387,49.799599,43.977268,38.948685,0.700708,0.926405,0.534213,0.864055,0.648505,0.883397
max,0.891057,0.896167,0.851909,108.330353,78.000000,92.023094,0.966551,0.990733,0.821468,0.994350,0.846126,1.000000


In [132]:
tta_combined_decoder_lr5_augs = pd.read_csv(tta_combined_decoder_lr5_augs_paths)
tta_combined_decoder_lr5_augs = clean_df(tta_combined_decoder_lr5_augs, True, 'brats')
tta_combined_decoder_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.242527,0.391863,0.364363,31.560822,36.003399,33.315482,0.202683,0.352967,0.293097,0.681073,0.629218,0.732325
std,0.220022,0.272252,0.279249,27.663382,27.969860,27.804451,0.244058,0.264630,0.246954,0.379408,0.301749,0.278996
min,0.000000,0.000000,0.000000,5.477226,5.000000,4.358899,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.008141,0.169323,0.047930,16.733200,20.000000,17.000000,0.004087,0.111048,0.024615,0.449302,0.476686,0.684901
50%,0.222639,0.317986,0.412681,26.000000,27.018513,23.618839,0.126616,0.382925,0.316170,0.877278,0.719315,0.799757
75%,0.340967,0.642334,0.601758,34.205261,44.147480,29.698484,0.228949,0.546013,0.513195,0.947374,0.871313,0.919173
max,0.619902,0.867626,0.746059,125.486252,114.004387,107.200745,0.857143,0.843632,0.682298,1.000000,0.929865,1.000000


In [133]:
tta_combined_decoder_lr4 = pd.read_csv(tta_combined_decoder_lr4_paths)
tta_combined_decoder_lr4 = clean_df(tta_combined_decoder_lr4, True, 'brats')
tta_combined_decoder_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.318397,0.530755,0.410040,32.494376,35.818402,29.804130,0.302959,0.807057,0.354958,0.541740,0.412017,0.662770
std,0.309062,0.205275,0.284159,26.350651,21.017304,27.941627,0.303414,0.240438,0.276543,0.424502,0.185932,0.271570
min,0.000000,0.000000,0.000000,3.605551,4.123106,3.741657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.017769,0.412636,0.101386,12.856844,20.469490,8.774964,0.009086,0.818464,0.056928,0.018265,0.278926,0.599517
50%,0.235647,0.551577,0.484391,25.416531,33.615471,18.920887,0.203912,0.911838,0.373222,0.597204,0.391348,0.698245
75%,0.619065,0.691448,0.654618,50.990196,52.478565,49.558048,0.552505,0.946781,0.569051,0.962212,0.549373,0.846076
max,0.874446,0.884012,0.847146,106.277687,78.892334,108.963295,0.885925,0.997611,0.870423,1.000000,0.818312,1.000000


In [134]:
tta_combined_decoder_lr4_augs = pd.read_csv(tta_combined_decoder_lr4_augs_paths)
tta_combined_decoder_lr4_augs = clean_df(tta_combined_decoder_lr4_augs, True, 'brats')
tta_combined_decoder_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.263963,0.483952,0.425441,24.061321,25.947841,22.945142,0.187178,0.465405,0.355037,0.843796,0.674700,0.813929
std,0.247003,0.275340,0.281685,12.818171,18.601559,20.525175,0.195500,0.296792,0.265756,0.233401,0.213823,0.164044
min,0.000000,0.040205,0.004714,5.830952,5.000000,3.000000,0.000000,0.020621,0.002363,0.000000,0.085019,0.463689
25%,0.032759,0.235677,0.177374,12.683616,12.385165,9.641419,0.016734,0.196267,0.098092,0.811849,0.564421,0.722392
50%,0.199761,0.543825,0.453882,24.000000,23.644093,17.227903,0.111367,0.433411,0.344826,0.921640,0.785318,0.884721
75%,0.417694,0.724174,0.673388,32.109173,33.670597,29.552360,0.300248,0.689828,0.584730,0.978322,0.820834,0.948970
max,0.668647,0.873554,0.854695,47.500511,78.209976,89.450546,0.556564,0.902246,0.765438,1.000000,0.892069,0.981864


In [135]:
tta_combined_first_layer_lr4_paths =    "/scratch-second/TTA_results/val_combined_first_layer_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_first_layer_lr4_augs_paths =    "/scratch-second/TTA_results/val_combined_first_layer_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_first_layer_lr5_paths =   "/scratch-second/TTA_results/val_combined_first_layer_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_first_layer_lr5_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_first_layer_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_first_layer_lr6_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_first_layer_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_first_layer_lr6_augs_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_first_layer_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [136]:
tta_combined_first_layer_lr4 = pd.read_csv(tta_combined_first_layer_lr4_paths)
tta_combined_first_layer_lr4 = clean_df(tta_combined_first_layer_lr4, True, 'brats')
tta_combined_first_layer_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.294995,0.525339,0.411039,31.261314,35.936531,31.901217,0.275568,0.812129,0.373278,0.551115,0.405162,0.626556
std,0.296117,0.210197,0.295561,23.534055,20.937485,30.734455,0.278583,0.250160,0.296067,0.435341,0.189408,0.310945
min,0.000000,0.000000,0.000000,4.000000,3.974014,3.741657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.386642,0.077060,10.099504,19.104973,7.874008,0.000000,0.825276,0.040495,0.000000,0.285180,0.509871
50%,0.154255,0.522661,0.491732,25.337719,33.837849,18.012482,0.187821,0.919488,0.423866,0.651777,0.368696,0.711811
75%,0.530675,0.677036,0.661680,48.435524,52.924473,56.053547,0.501730,0.944105,0.580744,0.991377,0.556099,0.890605
max,0.882382,0.889270,0.854534,98.285545,78.252159,107.067734,0.822770,0.996558,0.903567,1.000000,0.826402,1.000000


In [137]:
tta_combined_first_layer_lr4_augs = pd.read_csv(tta_combined_first_layer_lr4_augs_paths)
tta_combined_first_layer_lr4_augs = clean_df(tta_combined_first_layer_lr4_augs, True, 'brats')
tta_combined_first_layer_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.371635,0.568765,0.486560,20.025459,25.936431,24.430256,0.282906,0.565994,0.412474,0.821584,0.639420,0.821315
std,0.260330,0.234884,0.264572,13.222741,16.973871,21.247142,0.234647,0.248435,0.258403,0.256539,0.215942,0.134751
min,0.000000,0.127942,0.006845,5.830952,5.000000,4.000000,0.000000,0.108816,0.003435,0.000000,0.076602,0.545524
25%,0.121926,0.390647,0.255724,9.453603,13.500000,8.716990,0.066627,0.398511,0.165281,0.818427,0.555053,0.713345
50%,0.354437,0.648596,0.578313,16.278820,23.000000,17.114317,0.226969,0.594692,0.479613,0.950367,0.759343,0.872362
75%,0.605475,0.765869,0.697909,26.304283,35.671897,30.524635,0.459476,0.770703,0.592495,0.995524,0.775193,0.924760
max,0.836189,0.876175,0.769709,54.708317,69.831230,87.446564,0.841228,0.930627,0.813177,1.000000,0.836785,0.986743


In [138]:
tta_combined_first_layer_lr5 = pd.read_csv(tta_combined_first_layer_lr5_paths)
tta_combined_first_layer_lr5 = clean_df(tta_combined_first_layer_lr5, True, 'brats')
tta_combined_first_layer_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.366978,0.588825,0.389548,33.709173,27.205135,26.421881,0.379951,0.755239,0.323464,0.500814,0.500165,0.732172
std,0.305129,0.201054,0.281083,30.127911,21.652995,25.076093,0.310157,0.230031,0.270853,0.402157,0.189833,0.256523
min,0.000000,0.080340,0.000000,3.605551,4.000000,4.000000,0.000000,0.065141,0.000000,0.000000,0.080836,0.000000
25%,0.061911,0.469824,0.097119,9.035965,9.308866,9.339332,0.074630,0.676684,0.051227,0.032759,0.374037,0.668357
50%,0.355897,0.655456,0.424936,22.186658,19.874034,15.648222,0.352725,0.829409,0.312927,0.485683,0.519262,0.796995
75%,0.598139,0.730855,0.644879,54.331356,44.143708,34.000000,0.634934,0.924660,0.557825,0.879759,0.613473,0.898970
max,0.889264,0.893416,0.853542,98.593857,78.032043,92.023094,0.966551,0.991321,0.824010,1.000000,0.839736,1.000000


In [139]:
tta_combined_first_layer_lr5_augs = pd.read_csv(tta_combined_first_layer_lr5_augs_paths)
tta_combined_first_layer_lr5_augs = clean_df(tta_combined_first_layer_lr5_augs, True, 'brats')
tta_combined_first_layer_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.247155,0.408177,0.374886,29.474641,32.728439,30.990105,0.209697,0.371171,0.305765,0.718071,0.656220,0.771347
std,0.228449,0.274451,0.289550,25.958173,25.649443,26.790528,0.252483,0.264597,0.257810,0.356551,0.270677,0.228957
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.030163,0.161427,0.068670,16.798981,17.340573,15.806248,0.015382,0.155613,0.035765,0.610990,0.561921,0.699534
50%,0.228336,0.387672,0.417106,24.388542,26.108383,23.862321,0.130152,0.405739,0.308939,0.898799,0.712660,0.829147
75%,0.374562,0.653513,0.654508,33.563363,34.298964,31.063766,0.278256,0.554765,0.545045,0.953896,0.874659,0.921650
max,0.632488,0.866463,0.739218,125.486252,114.197197,107.373184,0.864817,0.840460,0.669822,1.000000,0.929991,1.000000


In [140]:
tta_combined_first_layer_lr6 = pd.read_csv(tta_combined_first_layer_lr6_paths)
tta_combined_first_layer_lr6 = clean_df(tta_combined_first_layer_lr6, True, 'brats')
tta_combined_first_layer_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.377754,0.574119,0.378340,34.389725,29.374619,27.686142,0.429745,0.773896,0.320131,0.452955,0.476643,0.681670
std,0.305455,0.199932,0.284665,30.491402,23.193346,24.742744,0.314649,0.221057,0.277814,0.385253,0.193541,0.281897
min,0.000000,0.131560,0.000000,4.472136,3.741657,4.000000,0.000000,0.121839,0.000000,0.000000,0.083920,0.000000
25%,0.041390,0.435497,0.067740,9.325486,10.456283,9.219060,0.089636,0.661597,0.035357,0.021302,0.345545,0.566264
50%,0.402452,0.599685,0.394900,23.697893,19.767261,17.716416,0.475956,0.846912,0.274501,0.363605,0.466303,0.736438
75%,0.625104,0.739546,0.620805,54.846245,48.396490,40.854301,0.705424,0.934999,0.539800,0.824254,0.618766,0.884193
max,0.890813,0.891389,0.847817,106.617050,78.364532,92.023094,0.967705,0.989352,0.857610,0.996047,0.831645,1.000000


In [141]:
tta_combined_first_layer_lr6_augs = pd.read_csv(tta_combined_first_layer_lr6_augs_paths)
tta_combined_first_layer_lr6_augs = clean_df(tta_combined_first_layer_lr6_augs, True, 'brats')
tta_combined_first_layer_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.273167,0.422024,0.352026,26.447789,31.784221,32.496521,0.230802,0.388599,0.289469,0.686775,0.666855,0.733345
std,0.228725,0.270433,0.301756,20.396169,19.956216,26.403662,0.251230,0.268120,0.273842,0.330698,0.255962,0.286086
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.043895,0.173103,0.028038,11.312284,20.250000,15.867011,0.023074,0.131931,0.014278,0.510868,0.548561,0.675089
50%,0.295467,0.441629,0.352087,22.687365,27.000000,27.771035,0.183547,0.432353,0.243291,0.830372,0.801625,0.791039
75%,0.489091,0.657264,0.606876,29.148717,37.243983,33.794327,0.351555,0.567998,0.513736,0.925794,0.841919,0.955015
max,0.663918,0.868258,0.798605,78.813705,90.887566,107.029198,0.895514,0.856425,0.786322,1.000000,0.924985,1.000000


In [142]:
tta_combined_norm_lr4_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr4/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_norm_lr4_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr4_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_norm_lr5_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr5/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_norm_lr5_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr5_augs/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_norm_lr6_paths =  "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr6/logs/BraTS_GLI/CCA_boxes.csv"
tta_combined_norm_lr6_augs_paths =  "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_combined_norm_lr6_augs/logs/BraTS_GLI/CCA_boxes.csv"

In [143]:
tta_combined_norm_lr4 = pd.read_csv(tta_combined_norm_lr4_paths)
tta_combined_norm_lr4 = clean_df(tta_combined_norm_lr4, True, 'brats')
tta_combined_norm_lr4.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000,46.000000
mean,0.277934,0.537179,0.394117,30.440064,34.090446,31.888387,0.261645,0.807323,0.352615,0.552638,0.421316,0.640388
std,0.303656,0.213836,0.294999,24.979197,21.251379,30.675548,0.283815,0.246750,0.289994,0.442078,0.197756,0.305532
min,0.000000,0.000000,0.000000,3.741657,3.741657,4.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.005712,0.390694,0.084385,10.236839,13.294534,8.613961,0.004603,0.812353,0.047294,0.004241,0.279454,0.568304
50%,0.131643,0.568152,0.474704,23.399722,33.226480,17.356380,0.119841,0.898447,0.406406,0.733862,0.412855,0.725838
75%,0.573774,0.727558,0.648541,49.995520,50.186753,54.356308,0.491718,0.947089,0.601821,0.987993,0.587584,0.878347
max,0.866444,0.892337,0.847780,105.990784,78.396431,108.300507,0.835473,0.995864,0.892825,1.000000,0.834284,1.000000


In [144]:
tta_combined_norm_lr4_augs = pd.read_csv(tta_combined_norm_lr4_augs_paths)
tta_combined_norm_lr4_augs = clean_df(tta_combined_norm_lr4_augs, True, 'brats')
tta_combined_norm_lr4_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000,20.000000
mean,0.319498,0.525632,0.436024,25.012816,29.677059,28.319008,0.245444,0.508499,0.363652,0.740614,0.650958,0.801416
std,0.266105,0.295510,0.284970,18.858087,20.100843,23.929521,0.241230,0.307314,0.270708,0.344810,0.238515,0.233222
min,0.000000,0.000000,0.000000,5.826631,5.000000,4.472136,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.024239,0.246785,0.151243,13.146534,14.500000,10.097171,0.012298,0.216010,0.082711,0.703825,0.603507,0.707365
50%,0.300861,0.643520,0.545907,21.102325,25.000000,22.000000,0.182272,0.591504,0.417123,0.861473,0.769153,0.885240
75%,0.565326,0.767574,0.683295,31.256689,39.347694,33.637225,0.408107,0.759936,0.592661,0.958292,0.800473,0.949327
max,0.836347,0.874006,0.818095,84.634506,73.940529,88.192970,0.841123,0.918400,0.798272,1.000000,0.856224,1.000000


In [145]:
tta_combined_norm_lr5 = pd.read_csv(tta_combined_norm_lr5_paths)
tta_combined_norm_lr5 = clean_df(tta_combined_norm_lr5, True, 'brats')
tta_combined_norm_lr5.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000,45.000000
mean,0.357243,0.583061,0.379157,33.288369,27.553272,28.305671,0.370825,0.744962,0.315333,0.495546,0.496624,0.724590
std,0.306748,0.203838,0.284275,30.336901,21.782772,27.101699,0.311889,0.234304,0.272748,0.405787,0.192012,0.255580
min,0.000000,0.072301,0.000000,3.605551,4.000000,4.012292,0.000000,0.062146,0.000000,0.000000,0.081549,0.000000
25%,0.054522,0.452690,0.084724,9.444536,9.486833,9.433981,0.069966,0.628600,0.044514,0.028052,0.363159,0.642673
50%,0.358724,0.641155,0.419744,22.022715,19.874607,17.117243,0.347460,0.821911,0.304013,0.538367,0.510725,0.788802
75%,0.583071,0.732666,0.644695,50.229473,44.553337,38.091995,0.633938,0.921967,0.552675,0.876289,0.648982,0.883098
max,0.889869,0.893990,0.852094,109.993034,78.025322,98.035706,0.966551,0.990805,0.823193,1.000000,0.841160,1.000000


In [146]:
tta_combined_norm_lr5_augs = pd.read_csv(tta_combined_norm_lr5_augs_paths)
tta_combined_norm_lr5_augs = clean_df(tta_combined_norm_lr5_augs, True, 'brats')
tta_combined_norm_lr5_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,0.237367,0.401213,0.360880,27.491057,31.642788,33.156004,0.199872,0.361985,0.294355,0.721116,0.665715,0.735240
std,0.222385,0.268443,0.290199,17.721420,21.537133,27.951984,0.246745,0.264216,0.258552,0.348193,0.266795,0.280512
min,0.000000,0.000000,0.000000,5.398965,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.035987,0.169880,0.047197,18.618532,18.000000,16.340136,0.018358,0.095101,0.024227,0.649682,0.593501,0.692512
50%,0.196257,0.310051,0.406527,25.000000,26.191601,26.683329,0.108939,0.377785,0.309961,0.879616,0.722610,0.835800
75%,0.337126,0.653962,0.644260,32.954514,36.565010,31.968735,0.221467,0.551326,0.533729,0.949387,0.869604,0.924723
max,0.635495,0.866124,0.739471,80.531670,91.454910,107.373184,0.855372,0.840742,0.672592,1.000000,0.928227,1.000000


In [147]:
tta_combined_norm_lr6 = pd.read_csv(tta_combined_norm_lr6_paths)
tta_combined_norm_lr6 = clean_df(tta_combined_norm_lr6, True, 'brats')
tta_combined_norm_lr6.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000,44.000000
mean,0.377161,0.574022,0.378571,34.655341,29.387179,27.672993,0.429453,0.773781,0.320806,0.452888,0.476605,0.680017
std,0.305234,0.199913,0.284958,31.011029,23.200926,24.750936,0.314476,0.221026,0.278324,0.385635,0.193576,0.282599
min,0.000000,0.131784,0.000000,4.472136,3.741657,4.000000,0.000000,0.122077,0.000000,0.000000,0.084044,0.000000
25%,0.041383,0.435461,0.067253,9.325486,10.447483,9.219060,0.088574,0.662389,0.035007,0.021298,0.345257,0.564897
50%,0.401757,0.598952,0.395658,23.687484,19.759610,17.716416,0.475509,0.847069,0.276228,0.362851,0.466125,0.735898
75%,0.622059,0.739563,0.620684,54.835991,48.386324,40.834575,0.704166,0.935122,0.542481,0.825291,0.618711,0.883553
max,0.890494,0.891415,0.848277,106.719254,78.492035,92.023094,0.967705,0.989233,0.857791,0.996000,0.831612,1.000000


In [148]:
tta_combined_norm_lr6_augs = pd.read_csv(tta_combined_norm_lr6_augs_paths)
tta_combined_norm_lr6_augs = clean_df(tta_combined_norm_lr6_augs, True, 'brats')
tta_combined_norm_lr6_augs.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000,24.000000
mean,0.279010,0.424284,0.365254,26.220393,31.232724,31.224244,0.236843,0.404495,0.299241,0.686580,0.663526,0.750598
std,0.236089,0.270916,0.295195,20.521016,20.364186,26.563720,0.255651,0.278743,0.268938,0.330386,0.260107,0.265948
min,0.000000,0.000000,0.000000,5.744563,5.000000,4.242640,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.043601,0.173167,0.028014,11.312284,17.413119,14.101772,0.022907,0.133321,0.014266,0.512437,0.546865,0.674096
50%,0.292468,0.481135,0.375699,22.963835,27.000000,25.652358,0.181270,0.447462,0.270310,0.832256,0.804215,0.789120
75%,0.487513,0.657101,0.606946,29.148717,37.243983,30.491795,0.383901,0.593333,0.515895,0.923608,0.841849,0.954812
max,0.662754,0.868229,0.798641,78.803558,90.727882,107.029198,0.894333,0.856207,0.788134,1.000000,0.924985,1.000000


In [149]:
mask = tta_combined_decoder_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_combined_decoder_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.409857,0.566438,0.423626,29.400205,36.875857,28.990268,0.380037,0.852481,0.375126,0.621478,0.442481,0.662901
std,0.323434,0.179705,0.268491,26.780157,21.665114,28.932163,0.305917,0.188252,0.275227,0.419825,0.165889,0.235611
min,0.000000,0.123991,0.000000,3.605551,7.000000,3.741657,0.000000,0.113216,0.000000,0.000000,0.067451,0.000000
25%,0.077659,0.482991,0.157423,9.219544,19.416489,7.348469,0.079508,0.846286,0.091925,0.168089,0.325908,0.604100
50%,0.347069,0.588110,0.484391,18.788294,33.955853,13.000000,0.358053,0.929060,0.404279,0.887296,0.425877,0.696414
75%,0.711039,0.743458,0.624413,41.048752,53.261150,38.183765,0.636762,0.946781,0.539888,0.980828,0.608918,0.804142
max,0.874446,0.781222,0.847146,106.277687,76.653442,108.963295,0.885925,0.992329,0.870423,1.000000,0.705688,1.000000


In [150]:
mask = tta_combined_first_layer_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.370681,0.570266,0.423441,29.229684,35.986486,28.388032,0.326703,0.865146,0.396229,0.601853,0.443951,0.670900
std,0.323160,0.183570,0.275714,25.323805,21.927972,28.984488,0.293274,0.192391,0.294691,0.442892,0.169049,0.261850
min,0.000000,0.069963,0.000000,4.000000,5.477226,3.741657,0.000000,0.050565,0.000000,0.000000,0.068308,0.000000
25%,0.000000,0.477343,0.146785,8.062258,18.493242,7.615773,0.000000,0.828449,0.079249,0.000000,0.315680,0.594800
50%,0.363064,0.612225,0.491732,22.036140,33.837849,13.652805,0.329368,0.934883,0.423866,0.879211,0.455141,0.711811
75%,0.621951,0.713617,0.661680,48.435524,53.488316,39.066597,0.570845,0.953764,0.619249,0.992658,0.600262,0.893430
max,0.882382,0.819159,0.846295,98.285545,76.752853,107.067734,0.822770,0.994299,0.903567,1.000000,0.732722,1.000000


In [151]:
mask = tta_combined_decoder_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_decoder_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.285153,0.445808,0.401105,23.426391,28.917394,26.264446,0.202097,0.429687,0.334911,0.888604,0.650699,0.822266
std,0.236270,0.278096,0.289425,12.599635,19.575067,21.889143,0.191616,0.284539,0.271532,0.116795,0.232700,0.149424
min,0.000908,0.040205,0.004714,5.830952,5.000000,3.741657,0.000454,0.020621,0.002363,0.622003,0.085019,0.544096
25%,0.068013,0.192577,0.161634,12.683616,14.000000,13.275612,0.035520,0.149586,0.088425,0.842543,0.531520,0.739354
50%,0.300738,0.464925,0.453882,24.000000,29.000000,21.047565,0.193236,0.433411,0.297107,0.921640,0.785318,0.884721
75%,0.417694,0.660534,0.673388,31.960406,36.664017,31.130474,0.300248,0.612395,0.584730,0.978322,0.803649,0.929123
max,0.668647,0.873554,0.757511,47.500511,78.209976,89.450546,0.556564,0.884426,0.737251,1.000000,0.892069,0.981864


In [152]:
mask = tta_combined_first_layer_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_first_layer_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.401494,0.559718,0.465930,20.569276,28.224077,28.083530,0.312237,0.559450,0.388076,0.875465,0.634014,0.821084
std,0.258489,0.237471,0.271972,14.435127,18.169090,22.569860,0.241217,0.239904,0.253952,0.143029,0.213186,0.138568
min,0.000194,0.127942,0.006845,5.830952,5.000000,4.000000,0.000097,0.108816,0.003435,0.520205,0.076602,0.545524
25%,0.222495,0.390647,0.232941,8.392147,15.030761,12.500000,0.134680,0.449481,0.151458,0.828723,0.555053,0.726461
50%,0.429188,0.648596,0.569401,16.524218,23.000000,24.062418,0.273784,0.594692,0.432522,0.950367,0.759343,0.872362
75%,0.605475,0.731637,0.669820,26.304283,40.441393,32.616570,0.478477,0.713340,0.592495,0.988744,0.767009,0.924760
max,0.836189,0.876175,0.769709,54.708317,69.831230,87.446564,0.841228,0.930627,0.780410,1.000000,0.832725,0.986743


In [153]:
mask = tta_combined_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.412909,0.618955,0.405738,33.973966,29.079811,28.527066,0.409849,0.793031,0.345480,0.501187,0.527615,0.719022
std,0.338740,0.185852,0.266066,30.821491,22.817152,27.216400,0.323650,0.207149,0.270257,0.418662,0.181041,0.210301
min,0.000000,0.084894,0.000000,4.000000,5.000000,4.000000,0.000000,0.065141,0.000000,0.000000,0.080836,0.000000
25%,0.048299,0.554823,0.164114,8.833402,8.760673,8.995772,0.048606,0.744589,0.096690,0.025756,0.431593,0.645445
50%,0.456242,0.676792,0.443753,21.635023,20.678196,13.890336,0.438488,0.858342,0.329534,0.521683,0.568571,0.761462
75%,0.701081,0.736368,0.640088,54.331356,46.993103,46.189178,0.706031,0.924814,0.578675,0.923339,0.648655,0.843528
max,0.889264,0.819234,0.853542,98.593857,78.032043,90.364822,0.905226,0.991321,0.824010,0.991118,0.818670,1.000000


In [154]:
mask = tta_combined_first_layer_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_first_layer_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,0.274616,0.434753,0.374387,29.670935,33.250300,32.271752,0.232997,0.400032,0.303871,0.797856,0.651256,0.752787
std,0.224374,0.274936,0.280613,27.431184,27.054645,27.863386,0.255928,0.262261,0.250697,0.273281,0.274681,0.233441
min,0.000000,0.000000,0.000000,5.099020,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.062047,0.201183,0.072024,13.627390,16.021719,16.619317,0.032072,0.186058,0.037612,0.749926,0.599124,0.693195
50%,0.260144,0.504020,0.417106,23.213294,25.938453,23.862321,0.157120,0.469816,0.308939,0.918787,0.712660,0.804385
75%,0.447399,0.659992,0.634260,34.258212,40.864641,31.831703,0.391273,0.564047,0.540060,0.963127,0.863705,0.903234
max,0.632488,0.866463,0.739218,125.486252,114.197197,107.373184,0.864817,0.840460,0.669822,1.000000,0.927263,1.000000


In [155]:
mask = tta_combined_norm_lr4['name'].isin(common_rows['name'])
filtered_rows = tta_combined_norm_lr4[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.368857,0.575926,0.413430,29.405241,34.682767,27.730401,0.319785,0.859601,0.378024,0.613093,0.453721,0.665742
std,0.328168,0.187399,0.269014,27.613603,21.877837,28.117972,0.291339,0.191156,0.282039,0.440304,0.178554,0.252728
min,0.000000,0.071908,0.000000,3.741657,5.099020,4.000000,0.000000,0.055736,0.000000,0.000000,0.066953,0.000000
25%,0.022847,0.476492,0.135106,8.414061,14.525839,7.874008,0.020381,0.835365,0.081309,0.016965,0.331294,0.590187
50%,0.296281,0.606146,0.479281,20.542639,33.196384,13.928389,0.344168,0.925121,0.408299,0.857719,0.465541,0.710211
75%,0.666213,0.734678,0.626220,50.549965,51.865211,34.203064,0.564725,0.949672,0.606254,0.988801,0.593876,0.813302
max,0.866444,0.823242,0.811728,105.990784,76.889534,108.300507,0.816429,0.991726,0.892825,1.000000,0.740145,1.000000


In [156]:
mask = tta_combined_norm_lr4_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_norm_lr4_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000,15.000000
mean,0.357463,0.500109,0.417369,25.343596,33.778679,33.653890,0.282684,0.483070,0.343150,0.812165,0.617356,0.783263
std,0.270519,0.291301,0.284920,20.695977,20.906617,25.264849,0.254664,0.298218,0.258329,0.259399,0.262419,0.260454
min,0.000000,0.000000,0.000000,5.826631,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.108587,0.225841,0.143224,10.849432,19.868608,17.645808,0.059299,0.197799,0.077794,0.831874,0.546035,0.706879
50%,0.352954,0.642722,0.546577,25.000000,30.000000,30.016663,0.233068,0.582944,0.402375,0.867362,0.766840,0.872487
75%,0.567470,0.722111,0.645739,31.780812,44.360430,44.430689,0.441867,0.707877,0.571486,0.940618,0.793606,0.940392
max,0.836347,0.874006,0.736510,84.634506,73.940529,88.192970,0.841123,0.918400,0.692188,1.000000,0.833706,1.000000


In [157]:
mask = tta_combined_decoder_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_decoder_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000,29.000000
mean,0.428995,0.625633,0.384955,33.371604,27.788163,30.358383,0.435838,0.793914,0.327731,0.512774,0.536141,0.683195
std,0.337360,0.181364,0.273316,32.198636,22.816281,28.144041,0.325980,0.201534,0.272690,0.406977,0.176553,0.258562
min,0.000000,0.125892,0.000000,4.000000,5.000000,4.000000,0.000000,0.098640,0.000000,0.000000,0.082733,0.000000
25%,0.063440,0.560125,0.154311,7.000000,8.774964,9.949874,0.103865,0.735282,0.090944,0.033876,0.438988,0.617425
50%,0.472215,0.681595,0.418688,20.904545,17.962454,15.264338,0.445126,0.861873,0.329787,0.622143,0.574598,0.735499
75%,0.703730,0.754130,0.635963,51.894123,46.150295,49.228039,0.758408,0.926405,0.574617,0.916321,0.655806,0.845897
max,0.891057,0.820111,0.851909,108.330353,78.000000,90.364822,0.902217,0.990733,0.821468,0.991476,0.823271,1.000000


In [158]:
mask = tta_combined_decoder_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_decoder_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.268056,0.415659,0.367806,31.966383,36.857041,34.353398,0.224018,0.378536,0.296471,0.752765,0.623023,0.711651
std,0.215789,0.274081,0.275624,29.124881,29.337133,29.066147,0.247260,0.264548,0.246457,0.320904,0.306999,0.284931
min,0.000000,0.000000,0.000000,5.477226,5.000000,4.358899,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.089748,0.182355,0.091970,14.449363,17.697402,16.654754,0.047670,0.120921,0.049306,0.696478,0.537589,0.663126
50%,0.263059,0.499788,0.412681,24.020824,27.018513,23.618839,0.152734,0.425795,0.316170,0.889336,0.719315,0.784058
75%,0.416252,0.650594,0.598253,37.158802,45.360003,41.616759,0.320438,0.555231,0.519058,0.961452,0.851532,0.913423
max,0.619902,0.867626,0.746059,125.486252,114.004387,107.200745,0.857143,0.843632,0.682298,1.000000,0.913031,1.000000


In [159]:
mask = tta_combined_norm_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_norm_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.407706,0.619530,0.404048,34.849556,28.744632,28.546380,0.406410,0.790038,0.344671,0.505743,0.529991,0.715171
std,0.339621,0.186849,0.266067,32.279307,22.912590,27.213566,0.324896,0.207486,0.270843,0.416307,0.182290,0.210824
min,0.000000,0.082809,0.000000,4.000000,5.000000,4.012292,0.000000,0.062708,0.000000,0.000000,0.081549,0.000000
25%,0.047987,0.554584,0.165241,8.833402,8.803270,8.995772,0.052475,0.733820,0.095718,0.025589,0.432972,0.638240
50%,0.453150,0.677942,0.446172,21.560644,19.696914,13.896791,0.431363,0.854882,0.332682,0.574028,0.570907,0.758204
75%,0.696625,0.751255,0.637895,54.399680,46.908637,46.193726,0.698222,0.923754,0.579184,0.919476,0.650056,0.837736
max,0.889869,0.819472,0.852094,109.993034,78.025322,90.364822,0.903391,0.990805,0.823193,0.991695,0.820671,1.000000


In [160]:
mask = tta_combined_first_layer_lr5['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr5[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000,28.000000
mean,0.412909,0.618955,0.405738,33.973966,29.079811,28.527066,0.409849,0.793031,0.345480,0.501187,0.527615,0.719022
std,0.338740,0.185852,0.266066,30.821491,22.817152,27.216400,0.323650,0.207149,0.270257,0.418662,0.181041,0.210301
min,0.000000,0.084894,0.000000,4.000000,5.000000,4.000000,0.000000,0.065141,0.000000,0.000000,0.080836,0.000000
25%,0.048299,0.554823,0.164114,8.833402,8.760673,8.995772,0.048606,0.744589,0.096690,0.025756,0.431593,0.645445
50%,0.456242,0.676792,0.443753,21.635023,20.678196,13.890336,0.438488,0.858342,0.329534,0.521683,0.568571,0.761462
75%,0.701081,0.736368,0.640088,54.331356,46.993103,46.189178,0.706031,0.924814,0.578675,0.923339,0.648655,0.843528
max,0.889264,0.819234,0.853542,98.593857,78.032043,90.364822,0.905226,0.991321,0.824010,0.991118,0.818670,1.000000


In [161]:
mask = tta_combined_norm_lr5_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_norm_lr5_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.262353,0.424181,0.358041,27.468221,32.090050,34.598185,0.220911,0.387493,0.290761,0.797023,0.662942,0.714424
std,0.219159,0.271132,0.282833,18.674234,22.646920,28.933206,0.250483,0.264357,0.252124,0.266174,0.269826,0.286739
min,0.000000,0.000000,0.000000,5.398965,5.000000,4.582576,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.068938,0.170559,0.056463,15.392029,16.692513,16.670068,0.035870,0.120738,0.029218,0.767135,0.605331,0.662467
50%,0.255684,0.496169,0.406527,23.757084,26.191601,26.683329,0.147736,0.449189,0.309961,0.905852,0.722610,0.785110
75%,0.414281,0.658513,0.623550,33.181297,40.418449,41.523554,0.320144,0.559148,0.523291,0.959909,0.864169,0.917328
max,0.635495,0.866124,0.739471,80.531670,91.454910,107.373184,0.855372,0.840742,0.672592,1.000000,0.915524,1.000000


In [162]:
mask = tta_combined_norm_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_combined_norm_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.428469,0.613072,0.398127,34.089511,30.905845,28.740009,0.459461,0.813382,0.350384,0.491110,0.512676,0.654741
std,0.335429,0.179848,0.268333,33.491811,25.016436,26.148774,0.327198,0.191514,0.279788,0.406918,0.183774,0.251901
min,0.000000,0.146848,0.000000,4.472136,4.242640,4.000000,0.000000,0.147558,0.000000,0.000000,0.084044,0.000000
25%,0.035623,0.555472,0.138563,8.405125,9.073544,9.611746,0.101018,0.766698,0.076969,0.018919,0.399460,0.563805
50%,0.431814,0.670916,0.403927,18.894444,19.519220,16.340136,0.503136,0.885044,0.314013,0.630449,0.548428,0.708357
75%,0.733439,0.747391,0.623943,56.874260,53.507910,44.961861,0.739237,0.936320,0.570934,0.870385,0.637598,0.805767
max,0.890494,0.813021,0.832503,106.719254,78.492035,87.074684,0.902770,0.989233,0.857791,0.991925,0.813975,1.000000


In [163]:
mask = tta_combined_norm_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_norm_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.283447,0.456364,0.355620,27.790285,33.075380,34.988159,0.245615,0.429521,0.282549,0.718167,0.660779,0.736531
std,0.206775,0.277057,0.280949,22.559854,22.209685,28.368748,0.256833,0.274837,0.242728,0.273348,0.266881,0.292527
min,0.000000,0.000000,0.000000,5.744563,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.081647,0.210634,0.019537,10.342686,16.826238,16.654754,0.042782,0.147177,0.009877,0.576735,0.587888,0.670705
50%,0.292779,0.514105,0.393998,22.561028,27.000000,27.622454,0.182168,0.448025,0.294347,0.823632,0.813500,0.790390
75%,0.475437,0.675674,0.599083,35.711870,42.572968,43.165951,0.355588,0.611249,0.489441,0.922950,0.836674,0.956780
max,0.641208,0.868229,0.735501,78.803558,90.727882,107.029198,0.894333,0.856207,0.658497,1.000000,0.886291,1.000000


In [164]:
mask = tta_combined_first_layer_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_combined_first_layer_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.429441,0.613193,0.397894,33.661508,30.888561,28.763177,0.460178,0.813641,0.349592,0.491301,0.512695,0.655828
std,0.335844,0.179904,0.268179,32.686843,25.005970,26.144452,0.327443,0.191494,0.279306,0.406778,0.183760,0.251686
min,0.000000,0.146734,0.000000,4.472136,4.242640,4.000000,0.000000,0.147328,0.000000,0.000000,0.083920,0.000000
25%,0.035647,0.555415,0.140221,8.405125,9.073544,9.661747,0.103043,0.767426,0.077955,0.018933,0.399439,0.565590
50%,0.433344,0.671123,0.402969,18.867962,19.534521,16.401220,0.505031,0.885514,0.312568,0.630393,0.548687,0.709730
75%,0.734430,0.747262,0.624113,56.881504,53.506845,44.978510,0.739832,0.936349,0.568464,0.870421,0.637715,0.806433
max,0.890813,0.812953,0.833208,106.617050,78.364532,87.074684,0.902735,0.989352,0.857610,0.992083,0.814135,1.000000


In [165]:
mask = tta_combined_first_layer_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_first_layer_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.283596,0.457253,0.354991,27.770458,33.088482,35.009053,0.246062,0.430195,0.281739,0.717862,0.660574,0.736867
std,0.207548,0.276349,0.280620,22.567332,22.233345,28.372383,0.257392,0.274119,0.242206,0.273375,0.266912,0.292561
min,0.000000,0.000000,0.000000,5.744563,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.080436,0.209189,0.019714,10.342686,16.826238,16.668535,0.042114,0.153588,0.009967,0.575747,0.588240,0.671634
50%,0.297197,0.514182,0.392547,22.615116,27.000000,27.631498,0.184052,0.448163,0.292778,0.823114,0.808656,0.794866
75%,0.475498,0.675565,0.599139,35.711870,42.577564,43.165951,0.357049,0.611362,0.488313,0.922923,0.836774,0.957098
max,0.642020,0.868258,0.735109,78.813705,90.887566,107.029198,0.895514,0.856425,0.657859,1.000000,0.885846,1.000000


In [166]:
mask = tta_combined_decoder_lr6['name'].isin(common_rows['name'])
filtered_rows = tta_combined_decoder_lr6[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.428119,0.612871,0.398120,33.689178,30.900790,28.740466,0.459307,0.813343,0.350606,0.490955,0.512408,0.654220
std,0.335176,0.179938,0.268494,32.679860,25.018495,26.146927,0.327060,0.191497,0.279895,0.406763,0.183945,0.252223
min,0.000000,0.146111,0.000000,4.472136,4.242640,4.000000,0.000000,0.148056,0.000000,0.000000,0.084113,0.000000
25%,0.035680,0.555301,0.137210,8.534532,9.044579,9.661747,0.099469,0.766915,0.076148,0.018952,0.398580,0.563187
50%,0.432760,0.670729,0.403735,18.894444,19.519220,16.340136,0.503757,0.885545,0.313901,0.631346,0.548239,0.707612
75%,0.733256,0.747400,0.624260,56.880892,53.484224,44.961613,0.739482,0.935957,0.571972,0.869631,0.637252,0.806489
max,0.890360,0.813047,0.833135,106.714561,78.497757,87.074684,0.902770,0.989280,0.857610,0.991801,0.813865,1.000000


In [167]:
mask = tta_combined_decoder_lr6_augs['name'].isin(common_augs['name'])
filtered_rows = tta_combined_decoder_lr6_augs[mask]
filtered_rows.describe()

,dice_NCR,dice_ED,dice_ET,hausdorff_NCR,hausdorff_ED,hausdorff_ET,recall_NCR,recall_ED,recall_ET,precision_NCR,precision_ED,precision_ET
count,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000,19.000000
mean,0.283055,0.456249,0.355612,27.761116,33.085229,34.984660,0.245361,0.429507,0.282588,0.719568,0.660550,0.724792
std,0.206704,0.277098,0.280995,22.554114,22.230462,28.370365,0.257065,0.274914,0.242897,0.271650,0.266785,0.285690
min,0.000000,0.000000,0.000000,5.744563,5.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.082068,0.209939,0.019601,10.342686,16.842194,16.577747,0.043015,0.146806,0.009909,0.579046,0.588606,0.671152
50%,0.292498,0.514307,0.394223,22.561028,27.000000,27.622454,0.181574,0.447987,0.294607,0.822944,0.809460,0.787222
75%,0.476085,0.675726,0.598935,35.332560,42.562138,43.165951,0.356637,0.611293,0.490209,0.923478,0.836634,0.924480
max,0.640481,0.868435,0.736447,78.813705,90.887566,107.029198,0.894923,0.857119,0.660622,1.000000,0.886236,1.000000


In [168]:
tta_ent_sup_lr4_90 = pd.read_csv(tta_ent_sup_lr4_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_90.describe()

NameError: name 'tta_ent_sup_lr4_90_paths' is not defined

In [ ]:
tta_ent_sup_lr4_augs_90 = pd.read_csv(tta_ent_sup_lr4_augs_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_90.describe()

In [ ]:
tta_ent_sup_lr4_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_90_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_90_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_augs_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_105_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_105/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_105_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr4_augs_105/logs/BraTS_GLI/SSA_boxes.csv"

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,41.000000,42.000000,43.000000,42.000000,44.000000,44.000000,43.000000,42.000000,43.000000
mean,21.500000,0.324426,0.497125,0.408297,32.355058,51.194817,56.864578,0.294769,0.781463,0.407793,0.595981,0.402535,0.569422
std,12.845233,0.308138,0.191550,0.251037,32.243460,15.291997,25.379832,0.297514,0.271435,0.304682,0.419757,0.149558,0.280593
min,0.000000,0.000000,0.000000,0.000000,3.605551,9.000000,5.099020,0.000000,0.000000,0.000000,0.000000,0.035061,0.000000
25%,10.750000,0.015720,0.422997,0.213118,8.602325,42.222553,42.430933,0.013383,0.679297,0.130416,0.046548,0.313816,0.447766
50%,21.500000,0.246090,0.550298,0.436968,17.888544,53.953918,58.525200,0.226739,0.903836,0.390697,0.825866,0.413747,0.566102
75%,32.250000,0.566836,0.629583,0.612216,51.734890,58.578131,70.178596,0.487501,0.962552,0.682605,0.951014,0.487892,0.776780
max,43.000000,0.879409,0.752073,0.814000,140.143677,89.044930,105.223564,0.942033,0.991484,0.967020,1.000000,0.773544,0.993976


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,41.000000,41.000000,41.000000
mean,20.000000,0.495372,0.552224,0.374596,34.682193,41.166470,38.642105,0.496585,0.648930,0.301344,0.574426,0.520424,0.705790
std,11.979149,0.287805,0.203896,0.255281,24.077199,15.180727,23.014303,0.320981,0.228718,0.248674,0.323001,0.212092,0.232278
min,0.000000,0.000000,0.128629,0.012580,2.828427,5.385165,4.898980,0.000000,0.099212,0.006342,0.000000,0.106339,0.047648
25%,10.000000,0.275319,0.376829,0.113588,10.168473,32.341923,19.709137,0.229131,0.482100,0.092131,0.409091,0.343021,0.592982
50%,20.000000,0.526514,0.599774,0.384361,35.589325,43.688663,42.953465,0.491021,0.714876,0.242304,0.645089,0.584136,0.769912
75%,30.000000,0.753613,0.712928,0.563877,51.253590,53.795910,53.361034,0.829358,0.825936,0.431440,0.808208,0.686962,0.876651
max,40.000000,0.877228,0.877293,0.865718,90.262360,63.765194,88.441513,0.934045,0.960371,0.853518,0.976514,0.844974,0.993000


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,37.000000,39.000000,39.000000,38.000000,39.000000,39.000000,38.000000,39.000000,39.000000
mean,19.000000,0.386572,0.542965,0.365341,38.783502,45.204024,40.426638,0.394172,0.640431,0.302453,0.558596,0.510062,0.634425
std,11.401754,0.304116,0.231474,0.274624,27.906069,18.382322,24.707967,0.326880,0.273146,0.260121,0.350189,0.219291,0.286275
min,0.000000,0.000000,0.009322,0.000038,3.743551,5.000000,2.236068,0.000000,0.006224,0.000019,0.000000,0.018557,0.000989
25%,9.500000,0.102245,0.361455,0.110163,11.357817,39.974083,18.917438,0.060110,0.435871,0.059939,0.235156,0.342246,0.546669
50%,19.000000,0.419198,0.615079,0.378527,39.759220,47.473675,39.912403,0.369220,0.742156,0.274954,0.643988,0.517968,0.691386
75%,28.500000,0.629550,0.709844,0.587565,60.235371,56.205624,55.817419,0.636559,0.857107,0.485970,0.866549,0.639528,0.822013
max,38.000000,0.886700,0.885929,0.865091,107.436958,77.181602,97.667801,0.956261,0.945964,0.847865,1.000000,0.893371,0.989008


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,41.000000,44.000000,44.000000,42.000000,44.000000,44.000000,43.000000,44.000000,44.000000
mean,21.500000,0.426725,0.525807,0.369579,39.723903,48.289712,44.860008,0.469006,0.659858,0.331832,0.556259,0.471305,0.658620
std,12.845233,0.306738,0.200150,0.277724,28.114911,17.681221,26.754283,0.321917,0.257015,0.284726,0.344086,0.203991,0.252798
min,0.000000,0.000000,0.052160,0.000000,4.000000,4.358899,3.000000,0.000000,0.037431,0.000000,0.000000,0.086001,0.000000
25%,10.750000,0.088582,0.375347,0.106223,13.390266,38.985639,23.975992,0.216573,0.485336,0.057087,0.315271,0.331787,0.555401
50%,21.500000,0.468481,0.547396,0.334102,35.336693,49.220453,45.990274,0.548757,0.742908,0.305611,0.657116,0.485404,0.743848
75%,32.250000,0.663840,0.687700,0.594036,63.085651,57.957726,64.234993,0.739435,0.848911,0.562630,0.839434,0.594763,0.816327
max,43.000000,0.863108,0.849721,0.864922,106.291107,91.766006,97.884621,0.973699,0.968088,0.840022,1.000000,0.886914,0.996899


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,36.000000,39.000000,39.000000,38.000000,39.000000,39.000000,37.000000,39.000000,39.000000
mean,19.000000,0.322056,0.468228,0.356997,39.619508,55.414338,60.800323,0.302170,0.737008,0.348697,0.604810,0.364251,0.525647
std,11.401754,0.339338,0.228671,0.265329,36.317059,21.272043,29.400952,0.325979,0.318879,0.304198,0.429736,0.175956,0.314997
min,0.000000,0.000000,0.000000,0.000000,3.741657,16.093477,6.082763,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9.500000,0.001815,0.315214,0.146654,9.933124,42.766579,42.939613,0.003302,0.662509,0.083675,0.014019,0.229551,0.308638
50%,19.000000,0.216511,0.546564,0.308053,30.854423,52.048054,62.968246,0.210127,0.908392,0.249156,0.878650,0.412475,0.571982
75%,28.500000,0.577423,0.631967,0.602424,53.159500,60.491379,76.210003,0.606261,0.957445,0.631970,0.961975,0.490514,0.791063
max,38.000000,0.897651,0.764826,0.827717,132.977448,117.226280,136.036758,0.876046,0.999101,0.958151,1.000000,0.625377,0.979173


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,41.000000,44.000000,44.000000,42.000000,44.000000,44.000000,43.000000,44.000000,44.000000
mean,21.500000,0.426725,0.525807,0.369579,39.723903,48.289712,44.860008,0.469006,0.659858,0.331832,0.556259,0.471305,0.658620
std,12.845233,0.306738,0.200150,0.277724,28.114911,17.681221,26.754283,0.321917,0.257015,0.284726,0.344086,0.203991,0.252798
min,0.000000,0.000000,0.052160,0.000000,4.000000,4.358899,3.000000,0.000000,0.037431,0.000000,0.000000,0.086001,0.000000
25%,10.750000,0.088582,0.375347,0.106223,13.390266,38.985639,23.975992,0.216573,0.485336,0.057087,0.315271,0.331787,0.555401
50%,21.500000,0.468481,0.547396,0.334102,35.336693,49.220453,45.990274,0.548757,0.742908,0.305611,0.657116,0.485404,0.743848
75%,32.250000,0.663840,0.687700,0.594036,63.085651,57.957726,64.234993,0.739435,0.848911,0.562630,0.839434,0.594763,0.816327
max,43.000000,0.863108,0.849721,0.864922,106.291107,91.766006,97.884621,0.973699,0.968088,0.840022,1.000000,0.886914,0.996899


In [ ]:
tta_ent_sup_lr5_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_90_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_90_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_105_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_105/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_105_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs_105/logs/BraTS_GLI/SSA_boxes.csv"

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_augs_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv'

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

FileNotFoundError: [Errno 2] No such file or directory: '/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv'

In [ ]:
tta_ent_sup_lr4_paths =    "/scratch-second/TTA_Augs/val_ent_con_decoder_lr5/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr4_augs_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_90_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr6_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_90_paths =    "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr6_augs_90/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_105_paths =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_105/logs/BraTS_GLI/SSA_boxes.csv"
tta_ent_sup_lr5_augs_105_paths  =   "/scratch/radjoe/Projects/Experiments/SAMEXP/logs/val_ent_con_decoder_lr5_augs_105/logs/BraTS_GLI/SSA_boxes.csv"

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_augs_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,15.000000,17.000000,16.000000,38.000000,39.000000,39.000000,15.000000,17.000000,16.000000
mean,19.000000,0.063585,0.149926,0.143806,23.833599,29.911321,24.496619,0.057451,0.131811,0.110964,0.596544,0.643774,0.836875
std,11.401754,0.145002,0.256374,0.262015,19.401745,19.547246,19.204773,0.165991,0.230503,0.211097,0.342630,0.313145,0.258427
min,0.000000,0.000000,0.000000,0.000000,5.123302,5.099020,3.605551,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,9.500000,0.000000,0.000000,0.000000,11.868098,16.763054,11.750000,0.000000,0.000000,0.000000,0.324864,0.520207,0.822930
50%,19.000000,0.000000,0.000000,0.000000,19.239279,21.000000,18.435925,0.000000,0.000000,0.000000,0.714801,0.818182,0.938911
75%,28.500000,0.028007,0.152634,0.122345,30.008331,46.108566,31.810621,0.014269,0.182423,0.066129,0.864222,0.862540,0.976989
max,38.000000,0.525128,0.867869,0.839296,77.110313,80.551529,75.520859,0.891972,0.873264,0.746731,1.000000,0.935860,1.000000


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_105_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,39.000000,38.000000,39.000000,39.000000,38.000000,39.000000,39.000000,38.000000,39.000000,39.000000,39.000000,39.000000,39.000000
mean,19.000000,0.410169,0.532364,0.370785,42.765808,51.721505,46.969735,0.369377,0.753988,0.311686,0.567900,0.434220,0.595418
std,11.401754,0.307857,0.194297,0.279057,29.827694,15.472049,25.491201,0.295212,0.247267,0.268481,0.347503,0.179040,0.260857
min,0.000000,0.000000,0.069047,0.000000,4.086168,6.164414,8.000000,0.000000,0.065198,0.000000,0.000000,0.067028,0.000000
25%,9.500000,0.148283,0.422048,0.080695,12.675951,44.468868,33.877609,0.086873,0.687205,0.044696,0.354960,0.325079,0.470231
50%,19.000000,0.377731,0.584855,0.423329,43.172459,53.319790,48.228622,0.363236,0.871681,0.311078,0.667805,0.463618,0.662329
75%,28.500000,0.671810,0.673033,0.616992,60.084005,60.662317,61.885786,0.628159,0.938834,0.519053,0.855575,0.545034,0.763628
max,38.000000,0.886159,0.816258,0.810157,107.327286,87.023849,100.756134,0.866565,0.980708,0.857610,0.993865,0.843925,0.961651


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_augs_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,44.000000,42.000000,44.000000,44.000000,24.000000,29.000000,28.000000,42.000000,44.000000,44.000000,24.000000,29.000000,28.000000
mean,21.500000,0.137766,0.254054,0.214791,24.922313,30.083382,29.821165,0.120581,0.225562,0.172575,0.676373,0.700221,0.752269
std,12.845233,0.202554,0.287027,0.292808,20.833364,15.670039,24.774945,0.217525,0.272644,0.252928,0.323161,0.252933,0.278527
min,0.000000,0.000000,0.000000,0.000000,5.744563,5.000000,3.741657,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.750000,0.000000,0.000000,0.000000,11.954865,21.000000,14.168420,0.000000,0.000000,0.000000,0.545861,0.582919,0.671955
50%,21.500000,0.001830,0.128528,0.020095,18.590545,27.000000,20.373440,0.000916,0.069185,0.010153,0.784217,0.809137,0.837299
75%,32.250000,0.308808,0.542816,0.397010,26.517834,38.238617,42.594537,0.186404,0.471647,0.292082,0.917938,0.883158,0.943188
max,43.000000,0.658163,0.868313,0.853820,78.390053,73.576820,89.699501,0.910272,0.853959,0.790047,1.000000,0.990469,1.000000


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr5_90_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

NameError: name 'pd' is not defined

In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_augs_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,41.000000,39.000000,41.000000,41.000000,12.000000,14.000000,13.000000,39.000000,41.000000,41.000000,12.000000,14.000000,13.000000
mean,20.000000,0.087257,0.132989,0.111739,23.638699,31.897608,31.232356,0.078906,0.116524,0.084923,0.692363,0.623940,0.744305
std,11.979149,0.200663,0.235474,0.227999,21.771418,20.304216,27.883270,0.199769,0.206052,0.181767,0.274584,0.317922,0.265639
min,0.000000,0.000000,0.000000,0.000000,3.162278,9.433981,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,10.000000,0.000000,0.000000,0.000000,10.060505,16.220486,12.000000,0.000000,0.000000,0.000000,0.565321,0.462851,0.701592
50%,20.000000,0.000000,0.000000,0.000000,13.641549,27.384348,21.189621,0.000000,0.000000,0.000000,0.797357,0.761749,0.791260
75%,30.000000,0.018463,0.153124,0.020480,34.004745,46.225486,43.462627,0.009331,0.105932,0.010375,0.883088,0.863760,0.942338
max,40.000000,0.718775,0.748789,0.731986,77.399605,79.655510,89.608604,0.795583,0.660589,0.602450,0.935811,0.928010,0.987021


In [ ]:
tta_ent_sup_lr4_augs_105 = pd.read_csv(tta_ent_sup_lr4_paths)
# tta_ent_sup_lr6_augs = clean_df(tta_ent_sup_lr6_augs, True, 'brats')
tta_ent_sup_lr4_augs_105.describe()

,Unnamed: 0,1,2,3,4,5,6,7,8,9,10,11,12
count,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,39.000000,41.000000,41.000000,41.000000,41.000000,41.000000
mean,20.000000,0.406915,0.534949,0.359984,36.555751,47.543220,42.150966,0.373015,0.751887,0.308079,0.575133,0.439231,0.600166
std,11.979149,0.301323,0.178649,0.276787,26.474480,16.015006,22.957735,0.283300,0.214401,0.269361,0.376947,0.176193,0.267227
min,0.000000,0.000000,0.072739,0.000000,3.741657,6.633250,5.696340,0.000000,0.073453,0.000000,0.000000,0.062226,0.000000
25%,10.000000,0.120204,0.476242,0.082412,11.565445,42.564655,32.878563,0.122263,0.637574,0.045752,0.253697,0.355629,0.484358
50%,20.000000,0.348877,0.541517,0.357642,36.496574,51.478149,42.043423,0.410366,0.815406,0.255484,0.731403,0.410388,0.629870
75%,30.000000,0.684825,0.682433,0.619893,56.081936,58.905018,57.148930,0.575549,0.928962,0.579297,0.903623,0.583292,0.787331
max,40.000000,0.851473,0.832973,0.805891,94.625053,70.021423,93.908463,0.854937,0.963256,0.752302,0.994186,0.758361,1.000000
